**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/jarzebowski-2026"], check=True)
    os.chdir("/content/encore/entries/jarzebowski-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Jarzebowski & Bendor (2026) on DANDI:000713

Paper: P. Jarzebowski and D. Bendor, "Temporal Gating by Chandelier Cells Encodes Signed Prediction
Errors", bioRxiv preprint, doi:10.64898/2026.06.26.734797. The claims file
(`claims/jarzebowski-2026.yaml`) was written from the preprint text (Europe PMC PPR1261135): main
text, STAR Methods and the legends of Figures 1 to 7. Figures S1 and S2 and their legends were not
in the text. The paper's code (github.com/bendor-lab/SETA-model_allen_analysis) was not public.

Dataset: DANDI:000713, Allen Institute Visual Behavior Neuropixels, version **0.240702.1725**
(doi:10.48324/dandi.000713/0.240702.1725). This is the version the paper's Methods cite, and the only
published version. All files are streamed.

Outside input: the Allen CCFv3 2017 annotation volume at 10 um, for the layer of each unit (Methods,
"Laminar classification"). It has no DOI; its SHA-256 is checked below.

| Claims | Data | Tested here |
|---|---|---|
| 14 model claims (Figures 1 to 4) | the model in the Methods (elsewhere) | all, simulated from the Methods |
| ccf-layer-link (added, link) | dataset and CCF annotation | yes |
| cell-counts and 14 claims on Figures 5 to 7, S1, S2 | dataset with the CCF layer (with_inputs) | all |

No claim is blocked.

Answered question. The first run asked a reader whether the Methods restrict the analysis to the
sessions of the optotagged mice (`questions/jarzebowski-2026.yaml`, cell-counts). The reader amended
the claims file: which sessions enter is now open detail O9, (a) all sessions or (b) only the
sessions of the Sst-IRES-Cre;Ai32 and VIP-IRES-Cre;Ai32 mice, and cell-counts and every claim that
takes its cells from it are run under both. This run shows both readings for every claim on the
recordings.

Revision after the first verification round. The verifier's report is answered as follows. Mixed
models that do not converge are refitted with other optimizers (`vbn.mixed`), and every table names
the optimizer used. Every intercept claim is run on all 16 cell selections of O1 to O3, under both O9
readings, and without the 10 sessions with a weak clock check (section before fig5c). The data check
prints the timing of every block, the optotagged units per genotype and O3 reading, the repetition
ordinals around an omission, and the running state per session. Figure 6 now adopts both
spontaneous blocks (see fig6b). The model sections add noise-free traces (fig2a), a refined
continuity test (fig2d) and 800 omission runs per gain (fig3b, fig3c).

Revision after the second verification round. The verifier agreed with every outcome. Text that did
not match the printed numbers is corrected (figs1 p ranges, the fig5c margins, the run time in the
README). The fig5c-fsb-highest-on-first-repetition section states how much the coding of repetition
moves the intercepts, and the fig6b section restates the reasons for the adopted spontaneous period,
with the block 1 alternative. The coding of repetition in fig5c is a new question to a reader.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = os.environ["OPENBLAS_NUM_THREADS"] = "1"   # many small model fits
import time, zlib, resource, warnings
import multiprocessing as mp
import numpy as np
import pandas as pd
from scipy import stats
import vbn
import seta_model as sm
warnings.simplefilter("ignore")
PAPER = "jarzebowski-2026"
print("dataset", vbn.DANDISET, "version", vbn.VERSION)

## Loading

Each of the 153 session files is streamed, and for its VISp units that pass the quality criteria
(D1, except the firing-rate bound, which has two readings) the spike count, burst-spike count and the
count in the 250 ms before are taken in every 250 ms window: active-task and passive image
presentations and omissions, spontaneous windows and optotagging pulses. Four sessions are read at a
time.

In [ ]:
t0 = time.time()
res, assets = vbn.load_all(workers=4)
print(f"streamed {len(res)} sessions in {time.time() - t0:.0f} s")
units_all = pd.concat([r["units"].assign(sess=i) for i, r in enumerate(res)], ignore_index=True)
t0 = time.time()
ccf_ids, ccf_sha, ccf_sizes = vbn.ccf_lookup(units_all[["x", "y", "z"]].values)
print(f"CCF annotation read in {time.time() - t0:.0f} s; sizes {ccf_sizes}; sha256 {ccf_sha}")
assert ccf_sha == vbn.CCF_SHA256, "the annotation volume differs from the one pinned"
layer_of = dict(zip(units_all.id.values, [vbn.LAYER_IDS.get(int(i), "other") for i in ccf_ids]))
ANN = [vbn.annotate(r) for r in res]
print("peak memory of this process (GB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6)

## Data check

In [ ]:
kinds = pd.Series(["session" if a["path"].count("_") == 1 else a["path"].rsplit("_", 1)[-1] for a in assets]).value_counts()
print("assets in the version:", len(assets), kinds.to_dict())
S = pd.DataFrame([dict(session=r["session"], subject=r["subject"], genotype=r["genotype"].split(";")[0],
                       image_set=r["image_set"].split("_")[6], novel=ANN[i].novel_session.iloc[0],
                       units_file=r["n_units_file"], visp_units_file=r["n_visp_units_file"], visp_d1=len(r["units"]),
                       presentations=int(r["pres"].active.sum()), omissions=int((r["pres"].omitted[r["pres"].active] == 1).sum()),
                       passive=int((~r["pres"].active).sum()), opto_pulses=len(r["opto"]),
                       speed_unit=r["speed_unit"], speed_min=r["speed_range"][0], speed_max=r["speed_range"][1])
                  for i, r in enumerate(res)])
print("sessions", len(S), "; subjects", S.subject.nunique(), "; sessions per subject", S.groupby("subject").size().value_counts().to_dict())
print("genotype:", S.genotype.value_counts().to_dict())
print("image set:", S.image_set.value_counts().to_dict(), "; novel sessions:", int(S.novel.sum()))
print(S[["units_file", "visp_units_file", "visp_d1", "presentations", "omissions", "passive", "opto_pulses"]].describe().round(1).to_string())
print("sessions with no VISp unit passing D1:", int((S.visp_d1 == 0).sum()))
print("running speed unit:", S.speed_unit.unique(), "; range", S.speed_min.min().round(1), "to", S.speed_max.max().round(1))
sb = pd.concat([r["spont_blocks"] for r in res]); sb["dur"] = sb.stop_time - sb.start_time
print("spontaneous blocks (s):"); print(sb.groupby("stimulus_block").dur.describe().round(2).to_string())
gap = [r["spont_blocks"].start_time.iloc[0] - r["active_span"][1] for r in res]
print("block 1 starts after the last active presentation by", np.round(np.min(gap), 2), "to", np.round(np.max(gap), 2), "s")
dur = np.concatenate([np.diff(r["pres"].start_time[r["pres"].active].values) for r in res])
print("onset-to-onset interval in the active task (s): median", np.median(dur).round(4), "; 1st to 99th percentile",
      np.percentile(dur, [1, 99]).round(3))
P = pd.concat(ANN)
print("Repetition ordinals present:", P[P.rep > 0].rep.value_counts().sort_index().loc[1:10].to_dict())
print("running state of active presentations (250 ms):", P.state_250.value_counts().to_dict(),
      "; without speed samples:", int(P.speed_250.isna().sum()))
print("unit columns, ranges over D1 VISp units:")
print(units_all[["isi_violations", "amplitude_cutoff", "presence_ratio", "snr", "firing_rate", "waveform_duration", "active_rate"]].describe().round(3).to_string())
# recordings deposited more than once: identical spike counts and first spike times
sig = pd.Series([(r["n_spikes_file"], tuple(np.round(r["first_spikes"], 6))) for r in res])
print("sessions sharing a spike-train signature with another:", int(sig.duplicated(keep=False).sum()))
print("subjects with more than one session sharing a start date:", int(S.assign(day=S.session.str[:8]).duplicated(["subject", "day"]).sum()))
print("CCF layer of D1 VISp units:", pd.Series(layer_of).value_counts().to_dict())

Clock check. With spikes and a stimulus table only, the population count of each session's VISp
units in the 250 ms after onset should tell image presentations from omissions, and the count in the
250 ms before onset should not. The area under the ROC curve is given for both windows, and for
30 to 100 ms after onset against the 70 ms before, as some sessions respond with a brief transient
followed by suppression, which the 250 ms count does not capture. The running
speed is checked only for coverage of the task (its timestamps are on the session clock by the
file's description); fig7b below is consistent with a shared clock.

In [ ]:
auc_post, auc_pre, auc_epost, auc_epre, auc_ses = [], [], [], [], []
for r, a in zip(res, ANN):
    if len(r["units"]) == 0:
        continue
    auc_ses.append(r["session"])
    om = a.omitted.values == 1
    img = ~om & ~a.post_om.values
    post = r["cnt"][:, a.i.values].sum(0).astype(float)
    pre = r["pre"][:, a.i.values].sum(0).astype(float)
    e_post = r["early"][:, a.i.values].sum(0).astype(float)
    e_pre = r["early_pre"][:, a.i.values].sum(0).astype(float)
    for x, store in [(post, auc_post), (pre, auc_pre), (e_post, auc_epost), (e_pre, auc_epre)]:
        u = stats.mannwhitneyu(x[img], x[om]).statistic
        store.append(u / (img.sum() * om.sum()))
auc_post, auc_pre, auc_epost, auc_epre = map(np.array, (auc_post, auc_pre, auc_epost, auc_epre))
print("sessions:", len(auc_post))
print("AUC image against omission, window after onset: median", np.median(auc_post).round(3), "min", auc_post.min().round(3))
print("AUC image against omission, window before onset: median", np.median(auc_pre).round(3), "range", auc_pre.min().round(3), "to", auc_pre.max().round(3))
print("sessions with AUC after onset below 0.8:", int((auc_post < 0.8).sum()))
print("30 to 100 ms after onset: median", np.median(auc_epost).round(3), "min", auc_epost.min().round(3),
      "; sessions below 0.8:", int((auc_epost < 0.8).sum()))
weak = [(s, round(x, 3), len(r["units"])) for s, x, r in zip(auc_ses, auc_epost, [r for r in res if len(r["units"])]) if x < 0.8]
print("sessions with AUC below 0.8 at 30 to 100 ms (session, AUC, VISp D1 units):", weak)
print("sessions without VISp units passing D1 (clock not checked here):", [r["session"] for r in res if len(r["units"]) == 0])
print("70 ms before onset: median", np.median(auc_epre).round(3), "range", auc_epre.min().round(3), "to", auc_epre.max().round(3))

In [ ]:
# Order and timing of the session's blocks; optotagged units; repetition ordinals; running state
B = pd.DataFrame([dict(session=r["session"], genotype=r["genotype"].split(";")[0], active_end=r["active_span"][1],
                       block1_start=r["spont_blocks"].set_index("stimulus_block").start_time.get(1),
                       block3_start=r["spont_blocks"].set_index("stimulus_block").start_time.get(3),
                       passive_start=r["pres"].start_time[~r["pres"].active].min(),
                       passive_block=int(r["pres"].stimulus_block[~r["pres"].active].iloc[0]),
                       active_block=int(r["pres"].stimulus_block[r["pres"].active].iloc[0])) for r in res])
B["block1_after_task"] = B.block1_start - B.active_end
B["block3_after_task"] = B.block3_start - B.active_end
B["order"] = np.where((B.active_end < B.block1_start) & (B.block1_start < B.block3_start) & (B.block3_start < B.passive_start),
                      "active (0), spont (1), spont (3), passive (5)", "other")
print("block order:", B.order.value_counts().to_dict(), "; active block", B.active_block.unique(), "; passive block", B.passive_block.unique())
print("seconds from the end of the active task to the start of each block:")
print(B[["block1_after_task", "block3_after_task"]].describe().round(1).to_string())
print("sessions where block 1 starts more than 60 s after the task:")
print(B[B.block1_after_task > 60][["session", "genotype", "active_end", "block1_start", "block3_start", "passive_start"]].round(1).to_string(index=False))
print("per session (s; first 10 rows, sorted by the gap to block 1):")
print(B.sort_values("block1_after_task", ascending=False)[["session", "genotype", "block1_after_task", "block3_after_task"]].head(10).round(1).to_string(index=False))

OT = pd.DataFrame([dict(session=r["session"], genotype=r["genotype"].split(";")[0], O3=o3,
                        tagged=int(vbn.opto_tagged(r, o3).sum()),
                        tagged_candidates=int((vbn.opto_tagged(r, o3) & (r["units"].firing_rate.values < 10) & (r["units"].firing_rate.values > 0.2)
                                               & (r["units"].waveform_duration.values > 0.4)
                                               & r["units"].id.map(layer_of).isin(["L2/3", "L5"]).values).sum()))
                   for r in res for o3 in "ab"])
print("optotagged D1 VISp units, by genotype and O3 reading (tagged_candidates: of them, L2/3 or L5 units that pass the rate and waveform tests):")
print(OT.groupby(["genotype", "O3"]).agg(sessions=("session", "nunique"), tagged=("tagged", "sum"),
      sessions_with_tagged=("tagged", lambda x: int((x > 0).sum())), max_per_session=("tagged", "max"),
      tagged_candidates=("tagged_candidates", "sum")).to_string())

a0 = ANN[0]
blk = a0[a0.omitted == 1].block.iloc[0]
ex = a0[a0.block == blk]
k = int(np.where(ex.omitted.values == 1)[0][0])
print(f"repetition ordinals around the first omission of session {res[0]['session']} (omitted flashes do not advance flashes_since_change):")
print(ex.iloc[max(k - 3, 0):k + 4][["start_time", "image_name", "flashes_since_change", "omitted", "is_change", "rep", "post_om"]].round(3).to_string())

OPTO_SES = {i for i, r in enumerate(res) if "Ai32" in r["genotype"]}   # O9 (b), as in cell-counts
RS = pd.DataFrame([dict(session=r["session"], O9b=i in OPTO_SES, running=(a.state_250 == "run").mean(),
                        stationary=(a.state_250 == "stat").mean(), rep1_running=int(((a.rep == 1) & (a.state_250 == "run")).sum()),
                        rep1_stationary=int(((a.rep == 1) & (a.state_250 == "stat")).sum()))
                   for i, (r, a) in enumerate(zip(res, ANN))])
print("fraction of active presentations by running state (250 ms), per session:")
print(RS[["running", "stationary", "rep1_running", "rep1_stationary"]].describe().round(3).to_string())
print("sessions with no stationary Repetition 1 presentation:", int((RS.rep1_stationary == 0).sum()), "; under O9 b:",
      int(((RS.rep1_stationary == 0) & RS.O9b).sum()), "of", int(RS.O9b.sum()))

## Model claims (Figures 1 to 4)

These claims are simulations of the paper's two-compartment model, which uses no recorded data. Their
data are "elsewhere": the model as the Methods specify it. The paper's code was not public, so the
model is written here from the Methods' equations and parameters (`seta_model.py`). Each claim's
generator is seeded from the claim, and each claim is also run under the seed plus 1000, 2000, 3000
and 4000.

In [ ]:
def seeds(cid):
    base = zlib.crc32(f"{PAPER}/{cid}".encode())
    return [base + k for k in (0, 1000, 2000, 3000, 4000)]

def sim(rng, runs=80, **kw):
    """Simulate and summarise; keyword arrays are broadcast into one table of conditions."""
    keys = list(kw)
    arrs = np.broadcast_arrays(*[np.atleast_1d(np.asarray(kw[k], float)) for k in keys])
    flat = {k: a.ravel() for k, a in zip(keys, arrs)}
    spk = sm.simulate(rng=rng, runs=runs, **flat)
    out = pd.DataFrame(sm.summarise(spk))
    for k in keys:
        out[k] = flat[k]
    return out, spk

def thresh(df, x, col="count", level=2.0):
    """Lowest x at which df[col] reaches level (inf when it never does)."""
    d = df.sort_values(x)
    hit = d[d[col] >= level]
    return float(hit[x].iloc[0]) if len(hit) else np.inf

def seed_table(cid, fn):
    rows = []
    for s in seeds(cid):
        r = fn(np.random.default_rng(s))
        rows.append(dict(seed=s, **{k: v for k, v in r.items() if np.isscalar(v)}))
    t = pd.DataFrame(rows)
    print(t.to_string(index=False))
    return t

L4_GRID = np.arange(0, 155, 5.0)       # claims file, fig2b criterion: 0 to 150 nS is a plotting choice
L4_WIDE = np.arange(0, 605, 5.0)       # extension used only to find thresholds above 150 nS

### model-positive-error-short-latency

> In the two-compartment model, L4 sensory input without L5 IT predictive input makes the L2/3
> neuron fire a short-latency singlet or doublet, depending on L4 strength, immediately after the
> input arrives; basket cell inhibition ends the response.

In [ ]:
def m_positive(rng):
    d, spk = sim(rng, l4=L4_GRID, l5=0.0, chc=0.0)
    i60 = int(np.where(L4_GRID == 60)[0][0])
    allspk = [t for run in spk[i60] for t in run]
    late = np.mean(np.array(allspk) > 8.5) if allspk else np.nan   # basket peak: 2.5 + 6 ms
    late_first_all = np.mean([run[0] > 8.5 for c in spk for run in c if run])   # first spikes, every L4
    late_spikes_all = np.mean([t > 8.5 for c in spk for run in c for t in run])  # all spikes, every L4
    p2max = d.p2.max()
    ok = (d.p1[i60] > 0.5) and (d.lat_median[i60] < 8.5) and (late == 0) and (p2max >= 0.5)
    return dict(p1_60=d.p1[i60], latency_60=d.lat_median[i60], frac_spikes_after_basket_peak_60=late,
                frac_first_spikes_after_peak_all_l4=late_first_all, frac_spikes_after_peak_all_l4=late_spikes_all,
                doublet_l4=thresh(d, "l4", "p2", 0.5), p2_max=p2max, criterion_met=ok, table=d)

r = m_positive(np.random.default_rng(seeds("model-positive-error-short-latency")[0]))
print(r["table"][["l4", "count", "p1", "p2", "lat_median"]].round(2).to_string(index=False))
print({k: v for k, v in r.items() if k != "table"})
late_by_l4 = [sum(t > 8.5 for run in c for t in run) for c in sm.simulate(l4=L4_GRID, l5=0.0, chc=0.0, runs=80,
              rng=np.random.default_rng(seeds("model-positive-error-short-latency")[0]))]
print("spikes after the basket peak (8.5 ms), by L4 (same draws):", dict(zip(L4_GRID.astype(int), late_by_l4)))
st = seed_table("model-positive-error-short-latency", m_positive)

**Result: partially reproduced, and sensitive to the seed.** With L4 at 60 nS and no prediction the
neuron fires in all 80 runs (p1 1.0), first spike a median 3.75 ms after L4 onset, and stronger L4
gives doublets (half the runs from 90 nS, all from 115 nS). Under the claim's seed 1 of the 81
spikes at 60 nS (0.012) falls after the basket conductance peak (8.5 ms), so the criterion's "no spike
after the basket peak" fails; under 3 of the 5 seeds there is none. Over every L4 from 0 to 150 nS,
0.009 of first spikes and 0.046 of all spikes fall after the basket peak under the claim's seed; the
late spikes are single spikes at weak L4 (15 to 30 nS, 18 spikes), second spikes of doublets (75 to
120 nS, 119 spikes) and 1 each at 60, 65 and 70 nS.

### model-negative-error-delayed-release

> With L5 IT predictive input and no sensory input, the ChC clamp holds the AIS below threshold and
> the neuron fires a single gated release spike after the clamp decays, 15 to 20 ms after the time the
> same L5 IT input could have triggered the L5 ET bAP.

Expected 17.75 ms, tolerance 4.25 (13.5 to 22 ms). Reading (a) latency from L5 IT onset; (b) from
the L5 IT conductance peak, 8 ms after onset.

In [ ]:
def m_negative(rng):
    d, spk = sim(rng, l4=0.0, l5=12.0, chc=25.0)
    a = d.lat_median[0]; b = a - 8.0
    single = d.single[0]
    return dict(p1=d.p1[0], latency_a=a, latency_b=b, single=single,
                ok_a=(13.5 <= a <= 22) and single == 1, ok_b=(13.5 <= b <= 22) and single == 1)

r = m_negative(np.random.default_rng(seeds("model-negative-error-delayed-release")[0]))
print(r)
st = seed_table("model-negative-error-delayed-release", m_negative)

**Result: reproduced.** With L5 IT 12 nS and ChC 25 nS and no L4, 97.5% of runs spike, every
spiking run fires a single spike, and the median latency from L5 IT onset is 17.4 ms (reading a), in
13.5 to 22 ms; from the L5 IT peak (reading b) it is 9.4 ms, outside. Over the five seeds reading (a)
gives 17.1 to 17.9 ms.

### fig2a-weak-sensory-silenced

> When L4 input is weaker than the predictive input, the ChC clamp vetoes the L2/3 spike and basket
> cell inhibition prevents the gated release spike, so the neuron is silent.

In [ ]:
WEAK = np.array([0, 2, 4, 6, 8, 10, 11.9])
def m_fig2a(rng):
    d, _ = sim(rng, l4=WEAK, l5=12.0, chc=25.0)
    weak = d[d.l4 > 0]
    ok = (weak["count"] < 0.1).all() and (weak["count"] < d["count"][0]).all()
    return dict(count_pred_only=d["count"][0], max_count_weak=weak["count"].max(),
                min_count_weak=weak["count"].min(), criterion_met=ok, table=d)

r = m_fig2a(np.random.default_rng(seeds("fig2a-weak-sensory-silenced")[0]))
print(r["table"][["l4", "count", "p1", "lat_median"]].round(3).to_string(index=False))
st = seed_table("fig2a-weak-sensory-silenced", m_fig2a)
# where, above 12 nS, the expected count is lowest (for reference)
d, _ = sim(np.random.default_rng(seeds("fig2a-weak-sensory-silenced")[0] + 1), l4=L4_GRID, l5=12.0, chc=25.0)
print("lowest expected count over 0 to 150 nS:", d["count"].min().round(3), "at L4 =", d.l4[d["count"].idxmin()], "nS")

In [ ]:
# Noise-free runs (sigma = 0) with the prediction (L5 IT 12 nS, ChC 25 nS): soma and AIS voltage and the
# four conductances, to show which input is left when the ChC clamp decays.
for l4 in [0.0, 10.0, 40.0]:
    tr = sm.trace(l4, 12.0, 25.0, sigma=0.0)
    tt = tr["t"]
    print(f"L4 {l4:.0f} nS: spikes at {tr['spikes']} ms; AIS peak before 15 ms {tr['v_ais'][(tt >= 0) & (tt < 15)].max():.2f} mV")
    for t0 in [2, 4, 6, 8, 10, 12, 14, 16, 18]:
        k = int(np.argmin(np.abs(tt - t0)))
        print(f"   t {t0:2d} ms  V_soma {tr['v_soma'][k]:7.2f}  V_AIS {tr['v_ais'][k]:7.2f}  g_L4 {tr['g_l4'][k]:5.1f}  "
              f"g_basket {tr['g_bc'][k]:5.1f}  g_L5IT {tr['g_l5'][k]:5.1f}  g_ChC {tr['g_chc'][k]:5.1f} nS")
# the L4 at which noise-free runs stop firing the release spike
fires = {x: len(sm.trace(x, 12.0, 25.0, sigma=0.0)["spikes"]) for x in np.arange(0, 62, 2.0)}
print("noise-free spike count by L4 (nS):", {int(k): v for k, v in fires.items()})

**Result: not reproduced.** With the prediction (L5 IT 12 nS, ChC 25 nS) and L4 below 12 nS, the
expected count is 0.90 to 0.975 per run, not below 0.1, and not lower than with the prediction alone
(0.975); the five seeds give 0.89 to 0.99. The model is quietest at L4 near 40 nS (0.15 spikes per
run), where L4 is stronger than L5 IT in conductance. Without noise the failure is the same: a release spike
fires at 18.7 ms for every L4 from 0 to 10 nS and none from 12 nS. At L4 10 nS the basket
conductance (1.2 times L4) peaks at 12 nS 8.5 ms after onset and has fallen to 6.4 nS at 18 ms, while L5 IT is
still 7.7 nS and the ChC conductance 4.5 nS, so the soma stays near -46.6 mV and the AIS reaches
threshold as the clamp decays. In this model the release spike is vetoed only when L4 is at least
as strong as L5 IT in conductance, the opposite of the claim's reading of "weaker"; a question to a
reader asks how the paper compares the two inputs.

### fig2b-breakthrough-doublet-threshold

> A sufficiently strong L4 input during the active ChC clamp overrides it and drives a breakthrough
> spike, but the L4 conductance needed for a doublet is higher than without a prediction.

In [ ]:
def m_fig2b(rng):
    p, _ = sim(rng, l4=L4_GRID, l5=12.0, chc=25.0)
    n, _ = sim(rng, l4=L4_GRID, l5=0.0, chc=0.0)
    out = dict(p1_pred_150=p.p1.iloc[-1],
               thr_a_pred=thresh(p, "l4", "count", 2), thr_a_none=thresh(n, "l4", "count", 2),
               thr_b_pred=thresh(p, "l4", "p2", 0.5), thr_b_none=thresh(n, "l4", "p2", 0.5))
    out["ok_a"] = out["p1_pred_150"] >= 0.5 and out["thr_a_pred"] > out["thr_a_none"]
    out["ok_b"] = out["p1_pred_150"] >= 0.5 and out["thr_b_pred"] > out["thr_b_none"]
    out["pred"], out["none"] = p, n
    return out

r = m_fig2b(np.random.default_rng(seeds("fig2b-breakthrough-doublet-threshold")[0]))
t = r["pred"][["l4", "count", "p2"]].merge(r["none"][["l4", "count", "p2"]], on="l4", suffixes=("_pred", "_none"))
print(t.round(2).to_string(index=False))
st = seed_table("fig2b-breakthrough-doublet-threshold", m_fig2b)
# threshold with prediction beyond the plotted range
w, _ = sim(np.random.default_rng(seeds("fig2b-breakthrough-doublet-threshold")[0] + 1), l4=L4_WIDE, l5=12.0, chc=25.0)
print("with prediction, doublet threshold over 0 to 600 nS: (a)", thresh(w, "l4", "count", 2), "(b)", thresh(w, "l4", "p2", 0.5))

**Result: reproduced.** With the prediction, every run spikes at L4 150 nS, and no L4 up to 150 nS
gives a doublet under either reading, while without it the doublet threshold is 110 nS (expected
count 2) or 90 nS (half the runs). Searched to 600 nS, the threshold with prediction is 345 nS (a) or
260 nS (b). All five seeds agree.

### fig2c-robust-to-input-delay

> Delaying either the predictive or the sensory input by 5 ms leaves the model's behaviour the same.

In [ ]:
DEL = np.arange(-5, 6, 1.0)
def m_fig2c(rng):
    d, _ = sim(rng, l4=60.0, l5=12.0, chc=25.0, sens_delay=np.maximum(DEL, 0), pred_delay=np.maximum(-DEL, 0))
    d["delay"] = DEL
    c0 = d["count"][d.delay == 0].iloc[0]
    dm = abs(d["count"][d.delay == -5].iloc[0] - c0); dp = abs(d["count"][d.delay == 5].iloc[0] - c0)
    return dict(count_0=c0, diff_minus5=dm, diff_plus5=dp, criterion_met=(dm < 0.5) and (dp < 0.5), table=d)

r = m_fig2c(np.random.default_rng(seeds("fig2c-robust-to-input-delay")[0]))
print(r["table"][["delay", "count", "p1", "lat_median"]].round(2).to_string(index=False))
st = seed_table("fig2c-robust-to-input-delay", m_fig2c)

**Result: reproduced.** At L4 60 nS with the prediction, the expected count is 0.775 at 0 ms delay,
and differs by 0.225 (prediction delayed 5 ms) and 0.3125 (sensory delayed 5 ms), both below 0.5.
The five seeds give differences of 0.125 to 0.3125.

### fig2d-latency-grows-as-sensation-weakens

> As sensory input weakens relative to the prediction, the latency of the breakthrough spike
> lengthens continuously.

"Without a single jump between two values" is tested as continuity, a choice made here (the
criterion gives no number). The interval of the largest 5 nS step is resampled at 1 nS with 4000
runs per conductance; a jump is a single 1 nS step that carries more than half the latency change
across that interval. A continuous curve spreads the change over the five steps, a discontinuity
puts it in one. The earlier tests (largest 5 nS step below half, or below a quarter, of the latency
range) are printed as well; with 80 runs per conductance their steps are limited by sampling noise
where few runs spike.

In [ ]:
def m_fig2d(rng):
    d, _ = sim(rng, l4=L4_GRID, l5=12.0, chc=25.0)
    s = d[d.n_spiking > 0].sort_values("l4")
    rho = stats.spearmanr(s.l4, s.lat_mean).statistic
    steps = np.abs(np.diff(s.lat_mean.values))
    rng_ = s.lat_mean.max() - s.lat_mean.min()
    # continuity: refine the interval of the largest step to 1 nS with 4000 runs per conductance; a jump
    # is one 1 nS step that carries more than half the latency change across that 5 nS interval
    j = int(np.argmax(steps)); lo, hi = s.l4.values[j], s.l4.values[j + 1]
    f, fspk = sim(rng, runs=4000, l4=np.arange(lo, hi + 0.01, 1.0), l5=12.0, chc=25.0)
    fsteps = np.abs(np.diff(f.lat_mean.values)); change_ = abs(f.lat_mean.values[-1] - f.lat_mean.values[0])
    lat = np.concatenate([[r[0] for r in c if r] for c in fspk])
    return dict(rho=rho, largest_step=steps.max(), latency_range=rng_, n_points=len(s),
                step_lt_half_range=steps.max() < 0.5 * rng_, step_lt_quarter_range=steps.max() < 0.25 * rng_,
                interval=f"{lo:.0f}-{hi:.0f}", fine_largest_step=fsteps.max(), fine_change=change_,
                fine_share=fsteps.max() / change_, frac_latency_8_12=np.mean((lat >= 8) & (lat < 12)),
                criterion_met=(rho < 0) and fsteps.max() < 0.5 * change_, table=s, fine=f)

r = m_fig2d(np.random.default_rng(seeds("fig2d-latency-grows-as-sensation-weakens")[0]))
print(r["table"][["l4", "n_spiking", "lat_mean"]].round(2).to_string(index=False))
print({k: v for k, v in r.items() if k not in ("table", "fine")})
print(r["fine"][["l4", "n_spiking", "lat_mean"]].round(2).to_string(index=False))
st = seed_table("fig2d-latency-grows-as-sensation-weakens", m_fig2d)

**Result: reproduced.** Over the 31 conductances where runs spike, the mean first-spike latency falls
from 18.4 ms at L4 0 to 2.36 ms at 150 nS (Spearman rho -0.999). The largest 5 nS step, 3.25 ms
between 35 and 40 nS, resampled at 1 nS with 4000 runs, changes latency by 3.03 ms in five steps of
which the largest is 1.30 ms (0.43 of the change), and 0.40 of the first spikes in that interval fall
between 8 and 12 ms, between the release and breakthrough latencies. The five seeds give rho -0.989
to -0.999 and a largest 1 nS share of 0.23 to 0.43, all below one half. The coarse test against a
quarter of the latency range fails under 3 of the 5 seeds (steps of 5.25, 6.28 and 6.06 ms of about
16 ms), which the 1 nS resampling shows to be sampling noise of the 80-run means, not a jump.

### fig3a-low-gain-suppresses-errors

> At low somatic gain of L4 and L5 IT, strong sensory input rarely drives bursts and an omission
> (prediction only) often ends in silence instead of a gated release spike.

In [ ]:
GAINS = np.array([0.2, 0.5, 1.0])
def m_fig3a(rng):
    g, l4 = np.meshgrid(GAINS, L4_GRID, indexing="ij")
    s, _ = sim(rng, l4=l4, l5=0.0, chc=0.0, gain=g)
    o, _ = sim(rng, l4=0.0, l5=12.0, chc=25.0, gain=GAINS)
    hi = s[s.gain == 1.0]
    strong = hi.l4[hi["count"] >= 2].values
    lo = s[(s.gain == 0.2) & s.l4.isin(strong)]
    hs = s[(s.gain == 1.0) & s.l4.isin(strong)]
    p2_low, p2_high = lo.p2.mean(), hs.p2.mean()
    silent_low = 1 - o.p1[o.gain == 0.2].iloc[0]
    return dict(n_strong=len(strong), strong_from=strong.min() if len(strong) else np.nan,
                p2_low=p2_low, p2_high=p2_high, silent_low=silent_low,
                silent_mid=1 - o.p1[o.gain == 0.5].iloc[0], silent_high=1 - o.p1[o.gain == 1.0].iloc[0],
                criterion_met=(p2_low < 0.5) and (p2_low < p2_high) and (silent_low > 0.5))

r = m_fig3a(np.random.default_rng(seeds("fig3a-low-gain-suppresses-errors")[0]))
print(r)
st = seed_table("fig3a-low-gain-suppresses-errors", m_fig3a)

**Result: reproduced.** High gain bursts (expected count >= 2) from L4 115 nS; at those 8
conductances no Low-gain run fires two spikes (fraction 0.0, High 1.0), and at Low gain all
prediction-only runs are silent (1.0). All five seeds agree.

### fig3b-gain-lowers-burst-threshold

> Raising L4 and L5 IT gain lowers the sensory threshold for bursts and raises the probability of a
> gated release spike on omission trials.

Readings (a) L5 IT and ChC kept (times the gain) while L4 varies, omission at L4 = 0; (b) sensory only
for the threshold, prediction only for the omission. Thresholds are searched over 0 to 600 nS, so
that one is found at each gain (the L4 range is a plotting choice).

In [ ]:
def m_fig3b(rng):
    g, l4 = np.meshgrid(GAINS, L4_WIDE, indexing="ij")
    a, _ = sim(rng, l4=l4, l5=12.0, chc=25.0, gain=g)
    b, _ = sim(rng, l4=l4, l5=0.0, chc=0.0, gain=g)
    o, _ = sim(rng, l4=0.0, l5=12.0, chc=25.0, gain=GAINS)
    out = {}
    for lab, d in [("a", a), ("b", b)]:
        th = [thresh(d[d.gain == x], "l4") for x in GAINS]
        om = [d[(d.gain == x) & (d.l4 == 0)].p1.iloc[0] for x in GAINS] if lab == "a" else list(o.p1)
        out.update({f"thr_{lab}_low": th[0], f"thr_{lab}_mid": th[1], f"thr_{lab}_high": th[2],
                    f"omit_{lab}_low": om[0], f"omit_{lab}_mid": om[1], f"omit_{lab}_high": om[2]})
        out[f"ok_{lab}"] = (th[0] > th[1] > th[2]) and (om[0] < om[1] < om[2])
    # omission probability from 800 prediction-only runs per gain (the omission run is the same under
    # both readings: L4 = 0 with the prediction), as 80 runs cannot resolve a probability near 0.03
    o8, _ = sim(rng, runs=800, l4=0.0, l5=12.0, chc=25.0, gain=GAINS)
    om8 = list(o8.p1)
    out.update(omit800_low=om8[0], omit800_mid=om8[1], omit800_high=om8[2])
    for lab in "ab":
        th = [out[f"thr_{lab}_{x}"] for x in ["low", "mid", "high"]]
        out[f"ok800_{lab}"] = (th[0] > th[1] > th[2]) and (om8[0] < om8[1] < om8[2])
    return out

r = m_fig3b(np.random.default_rng(seeds("fig3b-gain-lowers-burst-threshold")[0]))
print(r)
st = seed_table("fig3b-gain-lowers-burst-threshold", m_fig3b)

**Result: reproduced.** Under reading (b) the burst threshold falls from 515 nS (Low) to 230 (Mid) to
115 (High). With 800 prediction-only runs per gain the omission spike probability is 0.0, 0.026 and
0.948, so it rises from Low to Mid to High, and the criterion holds under all five seeds (Mid 0.013
to 0.040, Low 0 of 800 in every seed). With 80 runs, as before, Mid gain gave 0 to 6 spiking runs and
the strict rise failed under the claim's seed (0.0 and 0.0); 80 runs cannot resolve a probability
near 0.03. Under reading (a) no Low-gain threshold is found up to 600 nS and the Mid and High
thresholds are 350 and 355 nS, so the criterion fails there.

### fig3c-gain-keeps-release-delay

> Gain does not substantially change the delay of the gated release spike when it occurs.

In [ ]:
def m_fig3c(rng):
    o, _ = sim(rng, l4=0.0, l5=12.0, chc=25.0, gain=GAINS)
    lat = o.lat_median.values
    ok_vals = lat[np.isfinite(lat)]
    spread = ok_vals.max() - ok_vals.min() if len(ok_vals) > 1 else np.nan
    o8, _ = sim(rng, runs=800, l4=0.0, l5=12.0, chc=25.0, gain=GAINS)   # 800 runs, as in fig3b
    return dict(lat_low=lat[0], lat_mid=lat[1], lat_high=lat[2], n_low=o.n_spiking[0], n_mid=o.n_spiking[1],
                n_high=o.n_spiking[2], spread=spread, n_gains_with_spikes=len(ok_vals),
                criterion_met=(len(ok_vals) == 3) and spread < 5,
                n800_low=o8.n_spiking[0], n800_mid=o8.n_spiking[1], n800_high=o8.n_spiking[2],
                lat800_mid=o8.lat_median[1], lat800_high=o8.lat_median[2])

r = m_fig3c(np.random.default_rng(seeds("fig3c-gain-keeps-release-delay")[0]))
print(r)
st = seed_table("fig3c-gain-keeps-release-delay", m_fig3c)

**Result: not reproduced.** At Low gain no prediction-only run spikes, at Mid gain 1 of 80 does
(latency 24.0 ms) and at High gain 76 do (median 17.35 ms), so a release delay exists at only two
gains, 6.65 ms apart. Over the five seeds Mid gain has 1 to 4 spiking runs with latencies 22.1 to
34.8 ms against 17.35 to 18.2 ms at High. With 800 runs per gain, Low gain has 0 spiking runs in every
seed, Mid gain 18 to 24 runs with a median latency of 22.6 to 26.2 ms, and High gain 17.6 to 17.9 ms,
so the release delay still exists at only two gains (5.0 to 8.3 ms apart).

### fig4a-basket-reduction-lowers-burst-threshold

> Reducing basket cell conductance relative to L4 drive lowers the threshold for L2/3 bursts.

In [ ]:
BSC = np.array([0.6, 0.8, 1.0, 1.2])   # claims file: 0.6 to 1.2 in steps of 0.2, a plotting choice
def m_fig4a(rng):
    bs, l4 = np.meshgrid(BSC, L4_GRID, indexing="ij")
    d, _ = sim(rng, runs=10, l4=l4, l5=0.0, chc=0.0, bc_scale=bs)
    th = {x: thresh(d[d.bc_scale == x], "l4") for x in BSC}
    return dict(**{f"thr_{x}": v for x, v in th.items()}, criterion_met=all(th[x] < th[1.2] for x in BSC[:-1]))

r = m_fig4a(np.random.default_rng(seeds("fig4a-basket-reduction-lowers-burst-threshold")[0]))
print(r)
st = seed_table("fig4a-basket-reduction-lowers-burst-threshold", m_fig4a)

**Result: reproduced.** The burst threshold (mean count >= 2, 10 runs) is 55, 65, 85 and 105 nS at
basket scalings 0.6, 0.8, 1.0 and 1.2, each below 1.2's; all five seeds agree (1.2 gives 95 to 105 nS,
1.0 gives 75 to 90).

### fig4b-chc-reduction-shortens-latency

> Reducing ChC conductance shortens the latency of the first L2/3 spike for a given L4 strength and
> lowers the L4 needed for a positive error.

ChC conductance 0 to 50 nS in steps of 5 (a plotting choice; the range is not given). "The lowest L4
that drives a spike" is read as the breakthrough threshold: with the prediction present the neuron
fires a release spike at L4 = 0, so the threshold is the lowest L4 above the conductance of least
firing at which the expected count reaches 0.5 (an assumption; see the notes). Both this and the plain
lowest L4 > 0 with expected count >= 0.5 are printed.

In [ ]:
CHC = np.arange(0, 55, 5.0)
def m_fig4b(rng):
    c, l4 = np.meshgrid(CHC, L4_GRID, indexing="ij")
    d, _ = sim(rng, runs=10, l4=l4, l5=12.0, chc=c)
    # latency against ChC at each L4 where every ChC level has spiking runs
    rhos = []
    for x in L4_GRID:
        s = d[d.l4 == x]
        if (s.n_spiking > 0).all():
            rhos.append((x, stats.spearmanr(s.chc, s.lat_median).statistic))
    rhos = pd.DataFrame(rhos, columns=["l4", "rho"])
    th_b, th_plain = [], []
    for x in CHC:
        s = d[d.chc == x].sort_values("l4")
        imin = s["count"].values.argmin()
        after = s.iloc[imin:]
        hit = after[after["count"] >= 0.5]
        th_b.append(float(hit.l4.iloc[0]) if len(hit) else np.inf)
        pos = s[(s.l4 > 0) & (s["count"] >= 0.5)]
        th_plain.append(float(pos.l4.iloc[0]) if len(pos) else np.inf)
    rho_th = stats.spearmanr(CHC, th_b).statistic
    return dict(n_l4=len(rhos), n_rho_pos=int((rhos.rho > 0).sum()), min_rho=rhos.rho.min(),
                median_rho=rhos.rho.median(), rho_threshold=rho_th,
                criterion_met=bool((rhos.rho > 0).all()) and rho_th > 0,
                rhos=rhos, th=pd.DataFrame({"chc": CHC, "breakthrough": th_b, "lowest_l4_pos": th_plain}), table=d)

r = m_fig4b(np.random.default_rng(seeds("fig4b-chc-reduction-shortens-latency")[0]))
print(r["rhos"].round(2).T.to_string())
print(r["th"].T.to_string())
print({k: v for k, v in r.items() if np.isscalar(v)})
st = seed_table("fig4b-chc-reduction-shortens-latency", m_fig4b)

**Result: reproduced, resting on an assumption.** At all 16 L4 conductances where every ChC level
spikes, latency rises with ChC conductance (Spearman rho 0.46 to 0.99, median 0.95), and the
breakthrough threshold rises with ChC (rho 0.97, from 0 to 115 nS). The plain lowest L4 > 0 with
expected count >= 0.5 is 5 nS at every ChC level, because the release spike makes the neuron fire at
weak L4; the outcome rests on reading the paper's "L4 threshold required for generating a positive
error" as the breakthrough threshold. All five seeds agree.

### fig4cd-l5it-shortens-release-and-doublets

> Without sensory input, raising L5 IT conductance with ChC fixed at 25 nS shortens the release spike
> latency, and strong L5 IT input can produce a doublet instead of a singlet.

In [ ]:
L5G = np.arange(12, 42, 2.0)
def m_fig4cd(rng, grid=L5G):
    d, _ = sim(rng, l4=0.0, l5=grid, chc=25.0)
    s = d[d.n_spiking > 0]
    rho = stats.spearmanr(s.l5, s.lat_median).statistic
    dbl = thresh(d, "l5", "count", 2)
    return dict(rho=rho, lat_12=d.lat_median.iloc[0], lat_max=d.lat_median.iloc[-1], doublet_l5=dbl,
                max_count=d["count"].max(), criterion_met=(rho < 0) and np.isfinite(dbl), table=d)

r = m_fig4cd(np.random.default_rng(seeds("fig4cd-l5it-shortens-release-and-doublets")[0]))
if not np.isfinite(r["doublet_l5"]):   # claims file: extended if no doublet appears within 12 to 40 nS
    r = m_fig4cd(np.random.default_rng(seeds("fig4cd-l5it-shortens-release-and-doublets")[0]), np.arange(12, 122, 2.0))
print(r["table"][["l5", "count", "p2", "lat_median"]].round(2).to_string(index=False))
print({k: v for k, v in r.items() if np.isscalar(v)})
st = seed_table("fig4cd-l5it-shortens-release-and-doublets", lambda g: m_fig4cd(g, np.arange(12, 122, 2.0)))

**Result: reproduced.** Without L4, median release latency falls from 17.1 ms at L5 IT 12 nS to 7.7 ms
at 40 nS (rho -1.0), and the expected count reaches 2 at 22 nS. All five seeds agree (rho -0.9993 to
-0.9996, doublets from 24 nS on the 12 to 120 nS grid).

### fig4d-strong-prediction-lowers-burst-threshold

> With strong L5 IT drive, weak sensory inputs reach the burst threshold at a lower L4 conductance.

L5 IT 12 to 32 nS in steps of 4 (a plotting choice). Thresholds are searched up to 600 nS so that one
is found at 12 nS.

In [ ]:
L5D = np.arange(12, 36, 4.0)
def m_fig4d(rng):
    l5, l4 = np.meshgrid(L5D, L4_WIDE, indexing="ij")
    d, _ = sim(rng, l4=l4, l5=l5, chc=25.0)
    th = np.array([thresh(d[d.l5 == x], "l4") for x in L5D])
    rho = stats.spearmanr(L5D, th).statistic
    return dict(**{f"thr_{int(x)}": t for x, t in zip(L5D, th)}, rho=rho, criterion_met=rho < 0)

r = m_fig4d(np.random.default_rng(seeds("fig4d-strong-prediction-lowers-burst-threshold")[0]))
print(r)
st = seed_table("fig4d-strong-prediction-lowers-burst-threshold", m_fig4d)

**Result: reproduced.** With ChC at 25 nS the burst threshold is 355, 360, 330, 0, 0 and 0 nS for L5
IT 12 to 32 nS (rho -0.88); from 24 nS L5 IT alone gives doublets, so the threshold drops to 0. All
five seeds give rho -0.88 to -0.94. Most of the fall comes from L5 IT alone:
from 12 to 20 nS the threshold is 310 to 360 nS, it rises from 12 to 16 nS under 2 of the 5 seeds,
and it is lower at 20 than at 12 nS under all five (by 15 to 40 nS).

### fig4ef-compensation-restores-latency

> Raising the ChC conductance, or lowering the ChC reversal potential, can restore the release spike
> latency after an increase in L5 IT drive.

Prediction only, 10 runs per point (Methods). L5 IT raised to 20 nS (the claims file's choice), and
24 nS shown as well; ChC 25 to 100 nS in steps of 2.5 and E_ChC -58 to -80 mV in steps of 1 (plotting
choices).

In [ ]:
CG = np.arange(25, 102.5, 2.5)
EG = np.arange(-58, -81, -1.0)
def m_fig4ef(rng, l5=20.0):
    ref, _ = sim(rng, runs=10, l4=0.0, l5=12.0, chc=25.0)
    ref_lat = ref.lat_median[0]
    up, _ = sim(rng, runs=10, l4=0.0, l5=l5, chc=25.0)
    c, _ = sim(rng, runs=10, l4=0.0, l5=l5, chc=CG)
    e, _ = sim(rng, runs=10, l4=0.0, l5=l5, chc=25.0, e_chc=EG)
    cc = c[(c.chc > 25) & (abs(c.lat_median - ref_lat) <= 1)]
    ee = e[(e.e_chc < -58) & (abs(e.lat_median - ref_lat) <= 1)]
    return dict(ref_latency=ref_lat, raised_latency=up.lat_median[0],
                chc_restoring=cc.chc.min() if len(cc) else np.nan,
                echc_restoring=ee.e_chc.max() if len(ee) else np.nan,
                criterion_met=len(cc) > 0 and len(ee) > 0, c=c, e=e)

r = m_fig4ef(np.random.default_rng(seeds("fig4ef-compensation-restores-latency")[0]))
print({k: v for k, v in r.items() if np.isscalar(v)})
print(r["c"][["chc", "n_spiking", "lat_median"]].round(2).T.to_string())
print(r["e"][["e_chc", "n_spiking", "lat_median"]].round(2).T.to_string())
r24 = m_fig4ef(np.random.default_rng(seeds("fig4ef-compensation-restores-latency")[0] + 1), 24.0)
print("L5 IT 24 nS:", {k: v for k, v in r24.items() if np.isscalar(v)})
st = seed_table("fig4ef-compensation-restores-latency", m_fig4ef)

**Result: reproduced.** The reference latency is 18.2 ms; raising L5 IT to 20 nS shortens it to
11.65 ms, and ChC at 90 nS or E_ChC at -74 mV bring the median back within 1 ms. At L5 IT 24 nS the ChC
range up to 100 nS does not restore it, while E_ChC at -78 mV does. All five seeds meet the criterion
at 20 nS.

## Recorded data: layer link and cell selection

Common method (claims file, D1 to D9) is in `vbn.py`. Which sessions enter is open detail O9 of the
claims file: (a) all 153 sessions, with no unit removed by the optotagging filter in sessions of
other mice; (b) the 104 sessions of the Sst-IRES-Cre;Ai32 and VIP-IRES-Cre;Ai32 mice, where the
filter can be applied (Methods, "Identification of pyramidal neurons": candidates are filtered "by
removing putative inhibitory neurons identified in the optotagging data", which "used
VIP-IRES-Cre;Ai32 or Sst-IRES-Cre;Ai32 transgenic mouse lines"). Every claim below is run under
both, on the cells of cell-counts under its first-listed readings of O1 to O3 (O1 a a, O2 a, O3 a).
In the tables, `cells` "all" is O9 (a) and "opto" is O9 (b). Where the two give different outcomes,
O9 (b) is adopted as an assumption: it is the reading under which the Methods' optotagging filter
applies to every session, and the only one that gives the paper's cell and recording counts
(cell-counts). The section after
cell-counts runs the deciding test of each intercept claim on all 16 readings of O1 to O3.

### ccf-layer-link

> Layer labels from the Allen CCFv3 annotation volume (outside input) can be matched to the dataset's
> units through their CCF coordinates, and agree with the dataset's own area label.

In [ ]:
pts = units_all[["x", "y", "z"]].values
has_xyz = ~(pts == 0).all(1)
lay_all = units_all.id.map(layer_of)
print("D1 units labelled VISp in the files:", len(units_all), "in", units_all.sess.nunique(), "sessions")
print("with CCF coordinates:", has_xyz.sum(), "; all three coordinates 0:", (~has_xyz).sum(),
      "in", units_all.sess[~has_xyz].nunique(), "sessions")
inv = lay_all[has_xyz].isin(set(vbn.LAYER_IDS.values()))
print("units with coordinates whose voxel is a VISp layer:", inv.sum(), "of", has_xyz.sum(),
      f"({inv.mean():.4f})")
print("annotation ids of the others:", pd.Series(ccf_ids[has_xyz][~inv.values]).value_counts().to_dict())
print(lay_all[has_xyz].value_counts().to_string())
link_ok = inv.mean() >= 0.95
print("criterion (>= 95% of units with coordinates fall in a VISp layer):", link_ok)

**Result: reproduced.** Of the 8348 D1 units labelled VISp, 7997 have coordinates, and 7996 of them
(0.9999) fall in a VISp layer voxel; the other is in annotation id 484682512. 351 units in 7 sessions
have all three coordinates 0 and cannot be given a layer.

### cell-counts

> The analysis includes 266 putative pyramidal image-responsive L2/3 cells from 78 recordings and 1075
> such L5 cells from 98 recordings of mouse V1.

Every reading of O1 (sidedness, presentations), O2 (firing rate), O3 (optotagging baseline) and O9
(sessions), as the amended criterion has it.

In [ ]:
EXP = {"L2/3 cells": (266, 26.6), "L2/3 recordings": (78, 7.8), "L5 cells": (1075, 107.5), "L5 recordings": (98, 9.8)}
def count_row(c):
    g = c.groupby("layer").agg(cells=("unit", "size"), rec=("sess", "nunique"))
    v = {"L2/3 cells": g.cells.get("L2/3", 0), "L2/3 recordings": g.rec.get("L2/3", 0),
         "L5 cells": g.cells.get("L5", 0), "L5 recordings": g.rec.get("L5", 0)}
    v["all within"] = all(abs(v[k] - e) <= t for k, (e, t) in EXP.items())
    v["cells within"] = all(abs(v[k] - EXP[k][0]) <= EXP[k][1] for k in ["L2/3 cells", "L5 cells"])
    return v
OPTO_SES = {i for i, r in enumerate(res) if "Ai32" in r["genotype"]}
rows = []
for o1 in ["aa", "ab", "ba", "bb"]:
    for o2 in "ab":
        for o3 in "ab":
            c = vbn.select_cells(res, layer_of, o1, o2, o3)
            rows.append(dict(O9="a all", O1=o1, O2=o2, O3=o3, **count_row(c)))
            rows.append(dict(O9="b optotagged mice", O1=o1, O2=o2, O3=o3,
                             **count_row(c[c.sess.isin(OPTO_SES)])))
counts = pd.DataFrame(rows)
print(counts.to_string(index=False))
print("readings with all four counts within tolerance, O9 a (all sessions):", counts[counts.O9.str.startswith("a")]["all within"].sum(), "of 16")
print("same, O9 b (sessions of Sst-Cre and Vip-Cre Ai32 mice):", counts[counts.O9.str.startswith("b")]["all within"].sum(), "of 16")
print("sessions under O9 b:", len(OPTO_SES), "of", len(res))
CELLS = vbn.select_cells(res, layer_of, "aa", "a", "a")
CELLS_OPTO = CELLS[CELLS.sess.isin(OPTO_SES)].reset_index(drop=True)
print("cells used below, O9 a:", CELLS.groupby("layer").size().to_dict(), "; O9 b:", CELLS_OPTO.groupby("layer").size().to_dict(),
      "; subjects", CELLS.subject.nunique(), "and", CELLS_OPTO.subject.nunique())
print("subjects:", CELLS.subject.nunique(), "; per session, L2/3 cells median",
      CELLS[CELLS.layer == "L2/3"].groupby("sess").size().median(), ", L5 median", CELLS[CELLS.layer == "L5"].groupby("sess").size().median())

**Result: reproduced.** Under O9 (b), the 104 sessions of the Sst-Cre and Vip-Cre Ai32 mice, all 16
readings of O1 to O3 put the four counts within tolerance (245 to 270 L2/3 cells from 76 to 81
recordings; 996 to 1101 L5 cells from 100 recordings; first reading 247, 76, 997 and 100, against
266, 78, 1075 and 98). Under O9 (a), all 153 sessions, none does (363 to 396 L2/3 cells from 113 to
116 recordings; 1424 to 1577 L5 cells from 143).

In [ ]:
def percell(cells, fn):
    """One row per cell (or several), from fn(r, a, n, b, sn, sb) on the cell's counts."""
    out = []
    for c in cells.itertuples():
        r, a = res[c.sess], ANN[c.sess]
        got = fn(r, a, r["cnt"][c.unit], r["bst"][c.unit], r["scnt"][c.unit], r["sbst"][c.unit])
        if got is None:
            continue
        for d in (got if isinstance(got, list) else [got]):
            out.append(dict(session=c.session, subject=c.subject, sess=c.sess, layer=c.layer, unit=c.unit_id, **d))
    return pd.DataFrame(out)

def change(d, x, y, kind):
    """Per-cell change: 'diff' x - y; 'sqrt' sqrt(x) - sqrt(y) (O6 b); 'mi' modulation index (D5),
    0 when both are 0 (FSB) so that every cell with spikes counted stays in; NaN (left out)
    when a condition has no windows."""
    if kind == "diff":
        return d[x] - d[y]
    if kind == "sqrt":
        return np.sqrt(d[x]) - np.sqrt(d[y])
    mi = pd.Series(vbn.mod_index(d[x], d[y]), index=d.index)
    mi[(d[x] == 0) & (d[y] == 0)] = 0.0
    return mi

def fit(d, formula="y ~ 1", term=None, **lab):
    m = vbn.mixed(d, formula)
    row = m.loc[term] if term else m.iloc[0]
    return dict(**lab, coef=row.coef, t=row.t, df=int(row.df), p=row.p, ci_low=row.ci_low, ci_high=row.ci_high,
                n=int(row.n), sessions=int(row.groups), dropped=int(d.y.isna().sum()), converged=bool(row.converged),
                optimizer=row.optimizer)

def by_animal(d, sign):
    """Per-subject mean of the per-cell values; how many subjects go the claimed way."""
    s = d.dropna(subset=["y"]).groupby("subject").y.mean()
    return f"{(np.sign(s) == sign).sum()} of {len(s)} subjects have a mean in the claimed direction (range {s.min():.4f} to {s.max():.4f})"

def show(rows):
    t = pd.DataFrame(rows)
    with pd.option_context("display.float_format", lambda v: f"{v:.4g}"):
        print(t.to_string(index=False))
    return t

# Per-cell measures of the claims below
def f5(drop_post, mode):
    """Figure 5C: FSB, rate and window count at Repetition 1 and k = 2 to 8 (O8: drop_post, O4: mode)."""
    ix = {}
    def fn(r, a, n, b, sn, sb):
        out, v = [], {}
        if id(a) not in ix:   # the presentations of a session, found once for all its cells
            ix[id(a)] = {k: a[(a.rep == k) & (~a.post_om if drop_post else True)].i.values for k in range(1, 9)}
        for k in range(1, 9):
            m = ix[id(a)][k]
            v[k] = (vbn.fsb(n[m], b[m], mode), vbn.rate(n[m]), len(m))
        return [dict(k=k, f1=v[1][0], fk=v[k][0], r1=v[1][1], rk=v[k][1], n1=v[1][2], nk=v[k][2]) for k in range(2, 9)]
    return fn
SPB = {"block1": [1], "block3": [3], "both": [1, 3]}
def f6(r, a, n, b, sn, sb):
    om = a[a.omitted == 1].i.values
    sw = r["spontw"]
    d = {"ofr": vbn.rate(n[om]), "ofsb_a": vbn.fsb(n[om], b[om], "a"), "ofsb_b": vbn.fsb(n[om], b[om], "b"), "n_om": len(om)}
    for nm, blocks in SPB.items():
        for st in ["any", "stat"]:
            m = sw.stimulus_block.isin(blocks).values & ((sw.state_250.values == "stat") if st == "stat" else True)
            d[f"sfr_{nm}_{st}"] = vbn.rate(sn[m]) if m.any() else np.nan
            d[f"nwin_{nm}_{st}"] = int(m.sum())
            for o4 in "ab":
                d[f"sfsb_{nm}_{st}_{o4}"] = vbn.fsb(sn[m], sb[m], o4)
    return d
def f7(r, a, n, b, sn, sb):
    out = {}
    for w in ["250", "750"]:
        for cond, base in [("r1", a.rep == 1), ("om", a.omitted == 1)]:
            for st in ["run", "stat"]:
                ix = a[base & (a["state_" + w] == st)].i.values
                out[f"{cond}_{st}_{w}_n"] = len(ix)
                out[f"{cond}_{st}_{w}_spk"] = int(n[ix].sum())
                out[f"{cond}_{st}_{w}_fr"] = vbn.rate(n[ix]) if len(ix) else np.nan
                for o4 in "ab":
                    out[f"{cond}_{st}_{w}_{o4}"] = vbn.fsb(n[ix], b[ix], o4)
    return out
def fnov(r, a, n, b, sn, sb):
    if not a.novel_session.iloc[0]:
        return None
    nv = a[(a.rep == 1) & (a.img_block >= 1) & (a.img_block <= 10)].i.values
    fm = a[(a.rep == 1) & (a.img_block >= 20)].i.values
    if len(nv) == 0 or len(fm) == 0:
        return None
    d = {f"{k}_{m}": vbn.fsb(n[ix], b[ix], m) for k, ix in [("nov", nv), ("fam", fm)] for m in "ab"}
    d.update(nov_fr=vbn.rate(n[nv]), fam_fr=vbn.rate(n[fm]), n_nov=len(nv), n_fam=len(fm))
    return d

### Cell selection readings O1 to O3, and sessions with a weak clock check

Under O9 (b) all 16 readings of O1 to O3 give the paper's cell counts (cell-counts), so each is an
admissible selection. The claims below are run on the first (O1 a a, O2 a, O3 a); this section runs
the deciding test of each intercept claim on every selection, under both O9 readings, and once more
on the adopted cells without the 10 sessions whose clock check gave an AUC below 0.8. The per-cell
measures are those of the claims' sections, computed once on the union of the selections.

In [ ]:
t0 = time.time()
SEL = {o1 + o2 + o3: vbn.select_cells(res, layer_of, o1, o2, o3) for o1 in ["aa", "ab", "ba", "bb"] for o2 in "ab" for o3 in "ab"}
UNION = pd.concat(SEL.values()).drop_duplicates(["sess", "unit"]).reset_index(drop=True)
print("cells in any of the 16 selections (O9 a):", UNION.groupby("layer").size().to_dict())
PC = {nm: percell(UNION, fn) for nm, fn in [("f5aa", f5(False, "a")), ("f5ab", f5(False, "b")), ("f5ba", f5(True, "a")),
                                             ("f5bb", f5(True, "b")), ("f6", f6), ("f7", f7), ("nov", fnov)]}

def sub(tab, cells):
    k = set(zip(cells.sess, cells.unit_id))
    return tab[[x in k for x in zip(tab.sess, tab.unit)]]

def sel_outcomes(cells):
    """The deciding test of each intercept claim on one cell selection, as in the claim's section."""
    o, within5, best, nsig, slopes = {}, [], np.inf, 0, []
    for o8 in "ab":
        for o4 in "ab":
            d0 = sub(PC[f"f5{o8}{o4}"], cells)
            for o6 in ["linear", "sqrt"]:
                for code in ["k-2", "k"]:
                    v, sig, dist = {}, [], []
                    for lay in ["L2/3", "L5"]:
                        d = d0[d0.layer == lay].copy()
                        d["y"] = change(d, "fk", "f1", "diff" if o6 == "linear" else "sqrt")
                        d["rep"] = d.k - 2 if code == "k-2" else d.k
                        m = vbn.mixed(d, "y ~ rep")
                        c, p = m.loc["Intercept", "coef"], m.loc["Intercept", "p"]
                        e, tl = {"L2/3": (-0.06, 0.01), "L5": (-0.0505, 0.0045)}[lay]
                        v[lay] = c; sig.append(c < 0 and p < 0.05); dist.append(abs(c - e) - tl)
                        if lay == "L5" and code == "k-2":
                            slopes.append((m.loc["rep", "coef"], m.loc["rep", "p"]))
                    nsig += all(sig); best = min(best, max(dist))
                    if all(sig) and max(dist) <= 1e-12:
                        within5.append(f"O8 {o8} O4 {o4} {o6} {code}: L2/3 {v['L2/3']:.4f}, L5 {v['L5']:.4f}")
    o["fig5c-highest: readings neg. p<0.05 / within"] = f"{nsig}/16, {len(within5)}/16"
    o["fig5c-highest: closest miss"] = round(best, 4)
    o["fig5c-highest: readings within"] = "; ".join(within5)
    o["fig5c-l5-declining: L5 slope neg. p<0.05 within"] = f"{sum(c < 0 and p < 0.05 and abs(c + 0.0009) <= 0.0002 + 1e-12 for c, p in slopes)}/8"
    d6 = sub(PC["f6"], cells)
    r6 = []
    for lay in ["L2/3", "L5"]:
        d = d6[d6.layer == lay].copy(); d["y"] = vbn.mod_index(d.ofr, d.sfr_both_any)
        m = vbn.mixed(d).iloc[0]; r6.append(f"{lay} {m.coef:.3f} (p {m.p:.1e})")
    o["fig6b: both blocks, O5 a"] = "; ".join(r6)
    d = d6[d6.layer == "L5"].copy(); d["y"] = d.ofsb_a - d.sfsb_both_any_a
    m = vbn.mixed(d).iloc[0]; o["fig6c: both blocks, O5 a, O4 a, difference"] = f"{m.coef:.4f} (p {m.p:.2g})"
    d7 = sub(PC["f7"], cells)
    ra, rb = [], []
    for lay in ["L2/3", "L5"]:
        d = d7[(d7.layer == lay) & (d7.r1_run_250_n > 0) & (d7.r1_stat_250_n > 0)].copy()
        d["y"] = d.r1_run_250_a - d.r1_stat_250_a
        m = vbn.mixed(d).iloc[0]; ra.append(f"{lay} {m.coef:.3f} (p {m.p:.1e})")
        d["y"] = vbn.mod_index(d.r1_run_250_fr, d.r1_stat_250_fr)
        m = vbn.mixed(d).iloc[0]; rb.append(f"{lay} {m.coef:.3f} (p {m.p:.1e})")
    o["fig7a: 250 ms, O4 a, difference"] = "; ".join(ra)
    o["fig7b: 250 ms"] = "; ".join(rb)
    dn = sub(PC["nov"], cells); d = dn[dn.layer == "L5"].copy(); d["y"] = vbn.mod_index(d.nov_fr, d.fam_fr)
    m = vbn.mixed(d).iloc[0]; o["figs1-novel-rate: L5"] = f"{m.coef:.3f} (p {m.p:.2g})"
    return o

LOW_AUC = {s for s, _, _ in weak}
JOBS = [(k, o9) for k in SEL for o9 in "ba"] + [("aaaa", "b without low-AUC sessions")]
def run_job(job):
    k, o9 = job
    c = SEL[k] if o9 == "a" else SEL[k][SEL[k].sess.isin(OPTO_SES)]
    if o9.startswith("b without"):
        c = c[~c.session.isin(LOW_AUC)]
    return dict(O9=o9, O1=k[:2], O2=k[2], O3=k[3], **sel_outcomes(c))
with mp.get_context("fork").Pool(4) as pool:
    SELT = pd.DataFrame(pool.map(run_job, JOBS))
with pd.option_context("display.max_colwidth", 200, "display.width", 250):
    for col in SELT.columns[4:]:
        print(f"\n{col}:")
        print(SELT[["O9", "O1", "O2", "O3", col]].to_string(index=False))
print("\nsessions left out in the last row (AUC below 0.8):", sorted(LOW_AUC), "; of them under O9 b:",
      sum(r["session"] in LOW_AUC for i, r in enumerate(res) if i in OPTO_SES))
print(f"{time.time() - t0:.0f} s")

What the selections change. The selections add up to 422 L2/3 and 1697 L5 cells over all sessions.
No outcome below changes with the selection except that of fig5c-fsb-highest-on-first-repetition:
under O9 (b) its intercepts fall within tolerance under one analysis reading on 2 of the 16 selections
(O1 a b, O2 a, O3 a or b), by 0.0002 to 0.0003, and miss by 0.0001 to 0.0025 on the other 14. Under
every selection and both O9 readings, fig6b (both blocks) has both layers negative with L5 at -0.227
to -0.274, fig6c (both blocks) is -0.0050 to -0.0104 with p at most 0.021, fig7b has L5 at 0.170 to
0.199, and the novel against familiar L5 rate index is -0.035 to -0.068 with p at most 0.017. Leaving
out the 10 sessions with an AUC below 0.8 (8 of them in O9 b) changes no outcome.

### fig5c-fsb-highest-on-first-repetition

> In L2/3 and L5 neurons the fraction of spikes in bursts is highest for the first presentation of an
> image after a change and falls markedly from the first to the second repetition.

Expected intercepts L2/3 -0.06 (tolerance 0.01), L5 -0.0505 (0.0045). Readings: repetition coded k - 2
or k; O4 (a, b); O6 (linear, square root); O8 (a all presentations, b without those right after an
omission).

In [ ]:
F5 = {}
for cells_lab, cells in [("all", CELLS), ("opto", CELLS_OPTO)]:
    for o8 in "ab":
        for o4 in "ab":
            F5[(cells_lab, o8, o4)] = percell(cells, f5(o8 == "b", o4))
rows = []
for (cl, o8, o4), d0 in F5.items():
    for o6 in ["linear", "sqrt"]:
        for code in ["k-2", "k"]:
            for lay in ["L2/3", "L5"]:
                d = d0[d0.layer == lay].copy()
                d["y"] = change(d, "fk", "f1", "diff" if o6 == "linear" else "sqrt")
                d["rep"] = d.k - 2 if code == "k-2" else d.k
                rows.append(fit(d, "y ~ rep", "Intercept", cells=cl, O8=o8, O4=o4, O6=o6, coding=code, layer=lay))
                rows[-1]["slope"] = fit(d, "y ~ rep", "rep")["coef"]
                rows[-1]["slope_p"] = fit(d, "y ~ rep", "rep")["p"]
T5 = show(rows)
tol = {"L2/3": (-0.06, 0.01), "L5": (-0.0505, 0.0045)}
T5["within"] = [abs(c - tol[l][0]) <= tol[l][1] + 1e-12 for c, l in zip(T5.coef, T5.layer)]
T5["neg_sig"] = (T5.coef < 0) & (T5.p < 0.05)
g = T5.groupby(["cells", "O8", "O4", "O6", "coding"]).agg(both_neg_sig=("neg_sig", "all"), both_within=("within", "all"))
print(g.to_string())
for cl in ["all", "opto"]:
    print(f"readings (cells {cl}) with both negative, p < 0.05 and within tolerance:",
          int((g.loc[cl].both_neg_sig & g.loc[cl].both_within).sum()), "of", len(g.loc[cl]))
T5["dist"] = [abs(c - tol[l][0]) - tol[l][1] for c, l in zip(T5.coef, T5.layer)]
print("cells opto, distance outside tolerance (negative = inside), worst layer per reading:")
print(T5[T5.cells == "opto"].groupby(["O8", "O4", "O6", "coding"]).dist.max().round(4).to_string())
d0 = F5[("all", "a", "a")]
print("mean FSB by repetition (all sessions, O8 a, O4 a):")
print(pd.concat([d0[d0.k == 2].groupby("layer").f1.mean().rename(1)] +
                [d0[d0.k == k].groupby("layer").fk.mean().rename(k) for k in range(2, 9)], axis=1).round(4).to_string())
print("cells with no spikes at Repetition 1 (FSB set to 0):", int((d0[d0.k == 2].r1 == 0).sum()))
for cl in ["all", "opto"]:
    d0 = F5[(cl, "a", "a")]
    for lay in ["L2/3", "L5"]:
        d = d0[(d0.layer == lay) & (d0.k == 2)].assign(y=lambda x: x.fk - x.f1)
        print(cl, lay, "FSB(2) - FSB(1):", by_animal(d, -1))

**Result: reproduced, at the edge of the tolerance.** On the first cell selection (O1 a a, O2 a, O3 a)
every one of the 16 analysis readings gives negative intercepts with p far below 0.05 in both layers
under both O9 readings (O9 a: mean FSB 0.286 at Repetition 1 against 0.233 at Repetition 2 in L2/3,
0.209 against 0.163 in L5). Under O9 (b) none of the 16 puts both within tolerance there (closest,
O8 a, O4 a, square root, coding k, misses by 0.0024; first reading L2/3 -0.0444 and L5 -0.0448), and
38 of 49 subjects (L2/3) and 49 of 53 (L5) have a mean drop. The claim takes its cells from
cell-counts, whose 16 readings of O1 to O3 all give the paper's counts under O9 (b), and the criterion
asks for one reading. On the selections O1 a b, O2 a, O3 a or b (269 and 270 L2/3 cells, 1046 and
1045 L5 cells), one analysis reading (O8 a, O4 a, square root, coding k) puts both within tolerance:
L2/3 -0.0524 and L5 -0.0548 (L5 tolerance -0.055 to -0.046), inside by 0.0002 to 0.0003. The other 14
selections miss by 0.0001 to 0.0025. Under O9 (a) 8 of the 16 selections have a reading within
tolerance (first, O1 a a: O8 b, O4 a, linear, k - 2, L2/3 -0.0505, L5 -0.0473), inside by up to
0.0012. As cell-counts matches the paper only under O9 (b), the values recorded as measured are the
O9 (b) ones, L2/3 -0.0524 and L5 -0.0548. The intercepts sit at the edge of the paper's CIs: under
O9 (b) the outcome rests on one reading, inside the tolerance by 0.0002 to 0.0003. That reading codes
repetition as k, so its intercept is extrapolated to repetition 0 and is not the drop from Repetition
1 to 2 that the claim states; the claims file lists this coding as a reading. Coding k against k - 2
moves the L5 intercept by about 0.0018 (O9 b, O8 a, O4 a, square root: -0.0592 with k - 2 and
-0.0574 with k, above),
six times that margin. A question to a reader asks whether coding k fits the claim.

### fig5c-l5-fsb-keeps-declining

> After the drop at the second repetition, L5 FSB keeps decreasing at a low rate with further
> repetitions.

Expected slope -0.0009 per repetition, tolerance 0.0002.

In [ ]:
s = T5[T5.layer == "L5"][["cells", "O8", "O4", "O6", "coding", "slope", "slope_p"]].drop_duplicates(["cells", "O8", "O4", "O6"])
s["neg_sig"] = (s.slope < 0) & (s.slope_p < 0.05)
s["within"] = (s.slope - -0.0009).abs() <= 0.0002 + 1e-12
print(s.to_string(index=False))
rows = []
for (cl, o8, o4), d0 in F5.items():
    d = d0[d0.layer == "L5"].copy(); d["y"] = d.fk - d.f1; d["rep"] = d.k - 2
    rows.append(fit(d, "y ~ rep", "rep", cells=cl, O8=o8, O4=o4))
show(rows)

**Result: reproduced.** Under O9 (a) the L5 slope is -0.00089 per repetition (t(9973) = -2.91,
p = 0.0037, CI -0.0015 to -0.0003) under the first readings, within -0.0009 +/- 0.0002; under O9 (b)
it is -0.00099 (t(6977) = -2.67, p = 0.0077), also within. With O4 b the slope is about half (-0.00042
and -0.00051, p 0.04 to 0.05), and 3 of the 8 readings under O9 (a) and 4 of 8 under O9 (b) have p above 0.05.

### fig5c-l23-fsb-stable

> In L2/3, FSB does not change further with repetitions after the second.

A claim of absence: the slope of the Figure 5C model for L2/3, p < 0.05.

In [ ]:
s = T5[T5.layer == "L2/3"][["cells", "O8", "O4", "O6", "coding", "slope", "slope_p"]].drop_duplicates(["cells", "O8", "O4", "O6"])
print(s.to_string(index=False))
rows = []
for cl in ["all", "opto"]:
    d = F5[(cl, "a", "a")]; d = d[d.layer == "L2/3"].copy(); d["y"] = d.fk - d.f1; d["rep"] = d.k - 2
    rows.append(fit(d, "y ~ rep", "rep", cells=cl, test="paper's test, first readings"))
main = show(rows)

**Result: reproduced.** The L2/3 slope is -0.0004 per repetition under O9 (a) (t(2560) = -0.56,
p = 0.58, CI -0.0018 to 0.0010) and -0.0006 under O9 (b) (t(1727) = -0.70, p = 0.48); no reading of
either gives p below 0.05 (0.39 to 0.93).

### fig5d-firing-rate-dips-then-stable

> Firing rate falls at the second presentation, then recovers slightly to a stable level over further
> repetitions; neither L2/3 firing nor L5 bursting falls to zero.

In [ ]:
for (cl, o8) in [("all", "a"), ("all", "b"), ("opto", "a"), ("opto", "b")]:
    d = F5[(cl, o8, "a")].copy()
    d["mi"] = vbn.mod_index(d.rk, d.r1)
    med = d.groupby(["layer", "k"]).mi.median().unstack()
    print(f"cells {cl}, O8 {o8}: median modulation index of firing rate against Repetition 1")
    print(med.round(4).to_string())
    ok = all(med.loc[l, 2] < 0 and (med.loc[l, list(range(3, 9))] > med.loc[l, 2]).all() for l in ["L2/3", "L5"])
    fr23 = [d[(d.layer == "L2/3") & (d.k == 2)].r1.mean()] + [d[(d.layer == "L2/3") & (d.k == k)].rk.mean() for k in range(2, 9)]
    fsb5 = [d[(d.layer == "L5") & (d.k == 2)].f1.mean()] + [d[(d.layer == "L5") & (d.k == k)].fk.mean() for k in range(2, 9)]
    print("mean L2/3 rate (Hz), repetitions 1 to 8:", np.round(fr23, 3))
    print("mean L5 FSB, repetitions 1 to 8:", np.round(fsb5, 4))
    print("cells with both rates 0 (index undefined, left out):", int(d.mi.isna().sum()), "rows")
    print("criterion met:", ok and min(fr23) > 0 and min(fsb5) > 0)

**Result: reproduced.** Under O9 (a) the median modulation index against Repetition 1 is -0.386
(L2/3) and -0.209 (L5) at Repetition 2, and -0.18 to -0.25 and -0.10 to -0.13 at 3 to 8, all above
Repetition 2. Mean L2/3 rate stays at 3.5 to 7.5 Hz and mean L5 FSB at 0.157 to 0.209. O8 b and
O9 (b) meet the criterion as well (O9 b, O8 a: -0.394 and -0.207 at Repetition 2; L2/3 rate 3.2 to
6.9 Hz, L5 FSB 0.159 to 0.210).

### figs1-post-omission-bursting

> L5 bursting is higher for presentations that follow an omission than for those that precede it.

In [ ]:
def fs1(r, a, n, b, sn, sb):
    po, pr = a[a.post_om].i.values, a[a.pre_om].i.values
    return {f"{k}_{m}": vbn.fsb(n[ix], b[ix], m) for k, ix in [("post", po), ("pre", pr)] for m in "ab"}
rows = []
for cl, cells in [("all", CELLS), ("opto", CELLS_OPTO)]:
    D = percell(cells, fs1)
    for lay in ["L5", "L2/3"]:
        for o4 in "ab":
            for kind in ["diff", "mi", "sqrt"]:
                d = D[D.layer == lay].copy(); d["y"] = change(d, f"post_{o4}", f"pre_{o4}", kind)
                rows.append(fit(d, cells=cl, layer=lay, O4=o4, measure=kind))
    d = D[D.layer == "L5"].assign(y=lambda x: x.post_a - x.pre_a)
    print(cl, "L5, difference, O4 a:", by_animal(d, 1))
TS1 = show(rows)
for cl in ["all", "opto"]:
    print(f"L5 readings (cells {cl}) positive with p < 0.05:", int(((TS1.cells == cl) & (TS1.layer == "L5") & (TS1.coef > 0) & (TS1.p < 0.05)).sum()), "of 6")

**Result: reproduced.** L5 FSB after an omission exceeds that before it under all six readings and
both O9 readings: O9 (a) difference 0.0072, t(1424) = 5.32, p = 1.2e-07 under the first reading (p
2.7e-04 to 1.2e-07 over the others; 60 of 78 subjects); O9 (b) 0.0059, t(996) = 3.41, p = 6.8e-04 (p
0.025 to 6.8e-04; 37 of 53 subjects). L2/3, a variant, goes the same way (p 0.0026 to 0.032 under O9 a;
0.017 to 0.32 under O9 b).

### figs1-novel-image-bursting

> The first presentation of a newly introduced image drives more bursting than the first presentation
> of a familiar image.

Novel sessions (image set shown for the first time, `is_image_novel` set on some presentations);
Repetition 1 of the first 10 blocks of an image against blocks repeating it for the twentieth or later
time.

In [ ]:
NOV = {cl: percell(cells, fnov) for cl, cells in [("all", CELLS), ("opto", CELLS_OPTO)]}
print("novel sessions:", sum(ANN[i].novel_session.iloc[0] for i in range(len(res))), "of", len(res))
for cl in ["all", "opto"]:
    print("cells", cl)
    print(NOV[cl].groupby("layer").agg(cells=("unit", "size"), sessions=("sess", "nunique"),
          novel_presentations=("n_nov", "median"), familiar_presentations=("n_fam", "median")).to_string())
rows = []
for cl, D in NOV.items():
    for lay in ["L5", "L2/3"]:
        for o4 in "ab":
            for kind in ["diff", "mi", "sqrt"]:
                d = D[D.layer == lay].copy(); d["y"] = change(d, f"nov_{o4}", f"fam_{o4}", kind)
                rows.append(fit(d, cells=cl, layer=lay, O4=o4, measure=kind))
TNOV = show(rows)
for cl in ["all", "opto"]:
    d = NOV[cl][NOV[cl].layer == "L5"].assign(y=lambda x: x.nov_a - x.fam_a)
    print(cl, "L5, difference, O4 a:", by_animal(d, 1))

**Result: reproduced.** Under O9 (a), in the 66 novel sessions with L5 cells (637 cells), Repetition 1
FSB of an image's first 10 blocks exceeds that of its twentieth and later blocks under all six
readings (difference 0.036, t(636) = 6.04, p = 2.6e-09 under the first); under O9 (b), 46 sessions
and 453 cells, 0.033, t(452) = 5.35, p = 1.4e-07, all six readings with p below 0.005. In L2/3, a
variant, the difference is not significant (p = 0.18 under O9 a, 0.96 under O9 b), and only some
modulation-index and square-root readings under O9 (a) reach p < 0.05.

### figs1-novel-firing-rate-unchanged

> Firing rates do not differ between the first presentations of novel and familiar images.

A claim of absence: intercept of the per-cell modulation index (control familiar), p < 0.05.

In [ ]:
rows = []
for cl, D in NOV.items():
    for lay in ["L5", "L2/3"]:
        d = D[D.layer == lay].copy(); d["y"] = vbn.mod_index(d.nov_fr, d.fam_fr)
        rows.append(fit(d, cells=cl, layer=lay))
T1F = show(rows)
for cl in ["all", "opto"]:
    d = NOV[cl][NOV[cl].layer == "L5"].copy(); d["y"] = vbn.mod_index(d.nov_fr, d.fam_fr)
    print(cl, "L5:", by_animal(d, -1))

**Result: not reproduced.** Under O9 (b) the L5 modulation index of firing rate, novel against
familiar, is -0.043, t(452) = -2.61, p = 0.0095 (CI -0.075 to -0.010), so the paper's test finds a
difference; 30 of 46 subjects have a negative mean. Under O9 (a) it is -0.035, t(636) = -2.44,
p = 0.015 (CI -0.064 to -0.007). The earlier O9 (a) value (p = 0.059) came from a fit that had not
converged; refitted with Powell's method it converges and is significant. Over all 16 cell selections
and both O9 readings the index is -0.035 to -0.068 with p from 0.00027 to 0.017 (section before
fig5c), so the outcome no longer rests on O9. L2/3, a variant, shows none (0.019, p = 0.41 under O9 a;
-0.002, p = 0.94 under O9 b).

### fig6b-omission-firing-below-spontaneous

> Firing rate during stimulus omission is lower than during spontaneous activity, in both L2/3 and L5.

Expected L2/3 -0.17 (tolerance 0.06), L5 -0.115 (0.035). Which spontaneous block is meant is open: the
files hold a 10 s grey screen right after the active task (block 1) and a 289 s grey screen after the
receptive-field mapping (block 3); the data check shows that block 1 starts 18 to 29 s after the task in
152 sessions and block 3 about 940 s after it. Readings: block 1, block 3, both; each with all
windows (O5 a) or stationary windows only (O5 b).

Assumption: both blocks are adopted. The Methods name one spontaneous period, "the period of mouse
inactivity immediately following the task" (Methods, "Allen Brain Observatory data analysis"), and
the Results call it "spontaneous activity outside the stimulus blocks" (Results, "LTD-favouring
conditions"). Both blocks are grey-screen periods outside the stimulus blocks that follow the task,
and the files list both, and only these, in `intervals/spontaneous_presentations`. The mice run in
most windows of both blocks (the table below), so "inactivity" cannot mean stationary in either and
reads as the end of the task. Block 1 alone gives each cell 40 windows (10 s) and a median of 0
stationary windows, so O5 (b) cannot be applied to most cells. Under O5 (a), the adopted reading,
block 1 leaves out few cells (4 L2/3 and 15 L5 under O9 a, 1 and 9 under O9 b, the `dropped` column
below), so this argues against block 1 only under O5 (b). Against this, "immediately following" fits
block 1 alone, and the assumption is no better supported than block 1: both blocks give a median of
1155 block 3 windows per cell against 40 from block 1, so in effect they are block 3. Block 1 alone
is shown and does not reproduce either fig6b or fig6c; a question to a reader and one to the authors
ask which period was meant.

In [ ]:
SP = {cl: percell(cells, f6) for cl, cells in [("all", CELLS), ("opto", CELLS_OPTO)]}
D = SP["all"]
print("omission windows per cell, median:", D.n_om.median(), "; spontaneous windows, block 1:", D.nwin_block1_any.median(),
      ", block 3:", D.nwin_block3_any.median(), ", stationary block 1:", D.nwin_block1_stat.median())
SW = pd.concat([r["spontw"] for r in res])
print("running state of spontaneous windows, all sessions (fraction):")
print(SW.groupby("stimulus_block").state_250.value_counts(normalize=True).unstack().round(3).rename(columns={"": "neither"}).to_string())
print(D.groupby("layer")[["ofr", "sfr_block1_any", "sfr_block3_any", "ofsb_a", "sfsb_block1_any_a", "sfsb_block3_any_a"]].mean().round(4).to_string())
rows = []
for cl, D in SP.items():
    for nm in SPB:
        for st in ["any", "stat"]:
            for lay in ["L2/3", "L5"]:
                d = D[D.layer == lay].copy(); d["y"] = vbn.mod_index(d.ofr, d[f"sfr_{nm}_{st}"])
                rows.append(fit(d, cells=cl, spont=nm, O5=st, layer=lay))
T6 = show(rows)
tol = {"L2/3": (-0.17, 0.06), "L5": (-0.115, 0.035)}
T6["within"] = [abs(c - tol[l][0]) <= tol[l][1] for c, l in zip(T6.coef, T6.layer)]
T6["neg_sig"] = (T6.coef < 0) & (T6.p < 0.05)
print(T6.groupby(["cells", "spont", "O5"]).agg(both_neg_sig=("neg_sig", "all"), both_within=("within", "all")).to_string())
for cl in ["all", "opto"]:
    for lay in ["L2/3", "L5"]:
        d = SP[cl][SP[cl].layer == lay].copy(); d["y"] = vbn.mod_index(d.ofr, d.sfr_both_any)
        print(cl, lay, "both blocks:", by_animal(d, -1))

**Result: partially reproduced, resting on an assumption.** Against both spontaneous blocks (adopted)
with all windows (O5 a), omission firing is lower in both layers: L2/3 -0.221 (t(246) = -6.04,
p = 5.7e-09, CI -0.293 to -0.149) and L5 -0.228 (t(996) = -14.1, p = 4.3e-41) under O9 (b); L2/3
-0.245 and L5 -0.251 (p below 1e-15) under O9 (a). L2/3 is within its tolerance (-0.23 to -0.11)
under O9 (b), and L5 is outside its tolerance (-0.15 to -0.08) under both. Over all 16 cell
selections and both O9 readings both layers are negative with p below 1e-8 and L5 is -0.227 to -0.274,
always outside (section before fig5c); without the low-AUC sessions L5 is -0.223. 49 of 53 subjects
(L5) and 37 of 49 (L2/3) have a negative mean under O9 (b). With stationary windows only, L2/3 is
not significant (p = 0.32 and 0.45). Against block 1 alone, L5 is lower and within tolerance (O9 b
-0.112, p = 2.5e-08) but L2/3 is not lower (0.007, p = 0.87), which would not reproduce the claim;
block 3 alone gives the same outcome as both blocks.

### fig6c-omission-bursting-below-spontaneous

> L5 neurons show near-zero bursting during omission, with FSB below that of spontaneous activity.

Expected -0.0025, tolerance 0.0015. Readings: change measure (difference, modulation index), O4, O5,
O6 (linear, square root), and the spontaneous block as in fig6b.

In [ ]:
rows = []
for cl, D in SP.items():
    for nm in SPB:
        for st in ["any", "stat"]:
            for o4 in "ab":
                d = D[D.layer == "L5"].copy()
                for kind in ["diff", "mi", "sqrt"]:
                    d["y"] = change(d, f"ofsb_{o4}", f"sfsb_{nm}_{st}_{o4}", kind)
                    rows.append(fit(d, cells=cl, spont=nm, O5=st, O4=o4, measure=kind))
T6C = show(rows)
T6C["meets"] = (T6C.coef < 0) & (T6C.p < 0.05) & ((T6C.coef - -0.0025).abs() <= 0.0015)
T6C["neg_sig"] = (T6C.coef < 0) & (T6C.p < 0.05)
print(T6C.groupby(["cells", "spont"]).agg(readings=("meets", "size"), neg_sig=("neg_sig", "sum"), within=("meets", "sum")).to_string())
print("mean L5 FSB in omission windows (O4 a):", SP["all"].query("layer == 'L5'").ofsb_a.mean().round(4))

**Result: partially reproduced, resting on an assumption.** Against both spontaneous blocks (adopted),
O5 a, O4 a, difference, L5 omission FSB is below spontaneous: -0.0071, t(996) = -2.32, p = 0.020
(CI -0.0130 to -0.0011) under O9 (b), and -0.0050, p = 0.021 under O9 (a); outside the tolerance of
-0.004 to -0.001. Under O9 (b) 7 of the 12 readings against both blocks are negative with p < 0.05
and none is within tolerance (5 of 12 under O9 a). Over all 16 cell selections the first reading is
-0.0050 to -0.0104 with p at most 0.021 (section before fig5c). Against block 1 alone the difference is
positive (0.0073, p = 0.024 under O9 b) and no block 1 reading is negative with p < 0.05, which would
not reproduce the claim. Mean L5 omission FSB is 0.041, not near zero.

### figs2-l23-l5-activity-correlated

> Trial by trial, L5 population firing correlates with L2/3 population firing.

Sessions with at least 10 putative pyramidal cells per layer (Methods, "Unit inclusion"). The Methods
count putative pyramidal cells, without the responsive test; populations are formed (i) from the
cells of cell-counts and (ii) from all putative pyramidal cells. Reading (b) tests the per-session
Pearson coefficients with a one-sample t test, which is what an intercept-only model with one value
per session reduces to (the random effect cannot be told from the residual).

In [ ]:
PYR = vbn.select_cells(res, layer_of, "aa", "a", "a", responsive_only=False)
pc = PYR.groupby(["sess", "layer"]).size().unstack(fill_value=0)
S2_SES = list(pc[(pc["L2/3"] >= 10) & (pc["L5"] >= 10)].index)
print("putative pyramidal cells:", PYR.groupby("layer").size().to_dict(), "; sessions with >= 10 per layer:", len(S2_SES),
      "; of them in O9 b:", len([s for s in S2_SES if s in OPTO_SES]))
pr = CELLS.groupby(["sess", "layer"]).size().unstack(fill_value=0)
print("sessions with >= 10 responsive cells per layer:", int(((pr["L2/3"] >= 10) & (pr["L5"] >= 10)).sum()))
RESID = {}
for pop, cells in [("responsive", CELLS), ("pyramidal", PYR)]:
    for cond in ["rep1", "post"]:
        for fm in "ab":
            parts = []
            for s in S2_SES:
                cs = cells[cells.sess == s]
                if (cs.layer == "L2/3").sum() == 0 or (cs.layer == "L5").sum() == 0:
                    continue
                x = vbn.noise_residuals(res[s], ANN[s], cs, cond, fm)
                x["session"] = res[s]["session"]; x["sess"] = s
                parts.append(x)
            RESID[(pop, cond, fm)] = pd.concat(parts, ignore_index=True).rename(
                columns={"L2/3_fr": "x23", "L5_fr": "x5", "L5_fsb": "f5"})
rows = []
for (pop, cond, fm), x0 in RESID.items():
    if fm != "a":
        continue
    for cl in ["all", "opto"]:
        x = x0 if cl == "all" else x0[x0.sess.isin(OPTO_SES)]
        r_a = fit(x.assign(y=x.x5), "y ~ x23", "x23", cells=cl, population=pop, condition=cond, reading="a mixed model")
        rr = x.groupby("session").apply(lambda g: stats.pearsonr(g.x23, g.x5).statistic)
        tt = stats.ttest_1samp(rr, 0)
        rows.append(r_a)
        rows.append(dict(cells=cl, population=pop, condition=cond, reading="b per-session r", coef=rr.mean(), t=tt.statistic,
                         df=len(rr) - 1, p=tt.pvalue, n=len(rr), sessions=len(rr), median_r=rr.median()))
TS2 = show(rows)

**Result: reproduced.** In the 28 sessions with at least 10 putative pyramidal cells per layer
(O9 a), 21 of them in O9 (b), L5 and L2/3 rate residuals are positively related under both readings,
in both conditions and both populations (responsive cells, Repetition 1: slope 0.119, t(6806) = 16.0,
per-session r median 0.185, t(27) = 7.19 under O9 a; slope 0.106, t(5133) = 12.5, median r 0.164,
t(20) = 5.31 under O9 b; every p below 0.001). Only one session has 10 responsive cells per layer.

### figs2-l5-bursting-not-predicted-by-rate

> Trial by trial, L5 burst fraction is not predicted by L2/3 or L5 population firing rate, for the
> first repetition and for post-omission presentations.

A claim of absence; mixed models of the L5 FSB residual on the L2/3 rate residual and, separately, on
the L5 rate residual, p < 0.05. Readings (a) and (b) of which trials enter the L5 FSB mean.

In [ ]:
rows = []
for (pop, cond, fm), x0 in RESID.items():
    for cl in ["all", "opto"]:
        x = x0 if cl == "all" else x0[x0.sess.isin(OPTO_SES)]
        for pred in ["x23", "x5"]:
            rows.append(fit(x.assign(y=x.f5), f"y ~ {pred}", pred, cells=cl, population=pop, condition=cond, fsb_reading=fm,
                            predictor={"x23": "L2/3 rate", "x5": "L5 rate"}[pred]))
TS2B = show(rows)
print("cases with p < 0.05:", TS2B.groupby("cells").p.apply(lambda p: f"{int((p < 0.05).sum())} of {len(p)}").to_dict())

**Result: not reproduced.** Under both O9 readings the L5 rate residual predicts the L5 FSB residual
in every case (responsive cells, Repetition 1, slope 0.0103, t(6788) = 26.7 under O9 a and t(5116) =
23.1 under O9 b), and the L2/3 rate residual predicts it for Repetition 1 (p = 1.5e-05 and 6.3e-05),
though not for post-omission trials among responsive cells (p 0.38 to 1.0). 14 of 16 cases under O9
(a) and 13 of 16 under O9 (b) have p < 0.05.

### fig7a-running-raises-first-repetition-bursting

> Running increases L2/3 and L5 FSB on the first repetition of an image.

Expected L2/3 0.075 (0.025), L5 0.045 (0.015). Readings O4, O6, O7 (250 or 750 ms window for the
running state) and the change measure (difference, modulation index). Cells need Repetition 1
presentations in both states; the others are reported as dropped.

In [ ]:
LOC = {cl: percell(cells, f7) for cl, cells in [("all", CELLS), ("opto", CELLS_OPTO)]}
rows = []
for cl, D in LOC.items():
    for w in ["250", "750"]:
        for lay in ["L2/3", "L5"]:
            d0 = D[D.layer == lay]
            d = d0[(d0[f"r1_run_{w}_n"] > 0) & (d0[f"r1_stat_{w}_n"] > 0)].copy()
            for o4 in "ab":
                for kind in ["diff", "sqrt", "mi"]:
                    d["y"] = change(d, f"r1_run_{w}_{o4}", f"r1_stat_{w}_{o4}", kind)
                    rows.append(dict(fit(d, cells=cl, O7=w, layer=lay, O4=o4, measure=kind), no_both_states=len(d0) - len(d)))
T7A = show(rows)
tol = {"L2/3": (0.075, 0.025), "L5": (0.045, 0.015)}
T7A["within"] = [abs(c - tol[l][0]) <= tol[l][1] for c, l in zip(T7A.coef, T7A.layer)]
T7A["pos_sig"] = (T7A.coef > 0) & (T7A.p < 0.05)
g = T7A.groupby(["cells", "O7", "O4", "measure"]).agg(both_pos_sig=("pos_sig", "all"), both_within=("within", "all"))
print(g.to_string())
for cl in ["all", "opto"]:
    d = LOC[cl]; d = d[(d.layer == "L5") & (d.r1_run_250_n > 0) & (d.r1_stat_250_n > 0)].assign(y=lambda x: x.r1_run_250_a - x.r1_stat_250_a)
    print(cl, "L5, 250 ms, O4 a, difference:", by_animal(d, 1))

**Result: reproduced.** Under O7 250 ms, O4 a and the difference, running raises Repetition 1 FSB in
L2/3 (O9 a 0.071, t(274) = 4.55; O9 b 0.053, t(179) = 3.04) and L5 (O9 a 0.052, t(1119) = 5.97; O9 b
0.047, t(764) = 4.95), all within tolerance; the 750 ms window also puts both within tolerance under
both (O9 a 0.062 and 0.055; O9 b 0.059 and 0.058). The O4 b differences are smaller (L5 0.015, p =
0.0054 under O9 a; 0.012, p = 0.10 under O9 b). 91 L2/3 and 305 L5 cells under O9 (a), and 67 and 232 under O9 (b), lack
Repetition 1 presentations in one of the states and are left out. The paper's degrees of freedom (265 and 1074) imply that
every cell entered; under D7, 26 of the 104 O9 (b) sessions have no stationary Repetition 1
presentation (data check), so their cells cannot, and a question to a reader asks how the paper's n
arises. Over the 16 cell selections L2/3 is 0.049 to 0.059 and L5 0.047 to 0.049 under O9 (b) for
this reading.

### fig7b-running-raises-first-repetition-rate

> Running increases L2/3 and L5 firing rate on the first repetition of an image.

Expected L2/3 0.19 (0.10), L5 0.105 (0.055).

In [ ]:
rows = []
for cl, D in LOC.items():
    for w in ["250", "750"]:
        for lay in ["L2/3", "L5"]:
            d0 = D[D.layer == lay]
            d = d0[(d0[f"r1_run_{w}_n"] > 0) & (d0[f"r1_stat_{w}_n"] > 0)].copy()
            d["y"] = vbn.mod_index(d[f"r1_run_{w}_fr"], d[f"r1_stat_{w}_fr"])
            rows.append(fit(d, cells=cl, O7=w, layer=lay))
T7B = show(rows)
tol = {"L2/3": (0.19, 0.10), "L5": (0.105, 0.055)}
T7B["within"] = [abs(c - tol[l][0]) <= tol[l][1] for c, l in zip(T7B.coef, T7B.layer)]
print(T7B.groupby(["cells", "O7"]).agg(both_pos_sig=("p", lambda p: bool((p < 0.05).all())), both_within=("within", "all")).to_string())

**Result: partially reproduced.** Running raises Repetition 1 firing rate in L2/3 (O9 a 0.220,
p = 7.7e-11; O9 b 0.183, p = 7.2e-06; within tolerance) and L5 (O9 a 0.186, p = 2.1e-23; O9 b 0.174,
p = 1.3e-13), but L5 is above its tolerance (0.05 to 0.16) under both windows and both O9 readings
(750 ms: 0.191 and 0.198). Over all 16 cell selections and both O9 readings L5 is 0.170
to 0.199, above the tolerance under every one, and L2/3 0.181 to 0.253; without the low-AUC sessions
L5 is 0.173.

### fig7c-running-leaves-omission-bursting

> During omission, FSB does not differ between running and stationary conditions, in L2/3 or L5.

A claim of absence: FSB running minus stationary in omission windows, linear mixed model, p < 0.05.
Cell readings (a) omission windows in both states; (b) at least one omission spike in both states.
Expected counts 47 L2/3 cells from 35 recordings and 658 L5 cells from 94 recordings.

In [ ]:
def om_cells(d0, w, sel):
    d = d0[(d0[f"om_run_{w}_n"] > 0) & (d0[f"om_stat_{w}_n"] > 0)]
    if sel == "b":
        d = d[(d[f"om_run_{w}_spk"] > 0) & (d[f"om_stat_{w}_spk"] > 0)]
    return d.copy()
rows = []
for cl, D in LOC.items():
    for w in ["250", "750"]:
        for sel in "ab":
            for lay in ["L2/3", "L5"]:
                d = om_cells(D[D.layer == lay], w, sel)
                for o4 in "ab":
                    d["y"] = d[f"om_run_{w}_{o4}"] - d[f"om_stat_{w}_{o4}"]
                    rows.append(dict(fit(d, cells=cl, O7=w, selection=sel, layer=lay, O4=o4), recordings=d.sess.nunique()))
T7C = show(rows)
print("readings with p < 0.05 in some layer, by cells and selection:")
print(T7C.groupby(["cells", "selection"]).p.apply(lambda p: f"{int((p < 0.05).sum())} of {len(p)}").to_string())

**Result: reproduced, resting on an assumption.** Under O9 (b), adopted, no layer, window or O4
reading gives p < 0.05 under either cell selection (selection b, 250 ms, O4 a: L2/3 -0.032, p = 0.13;
L5 -0.0005, p = 0.91; lowest p 0.059). Its counts are 143 L2/3 cells from 60 recordings and 778 L5
cells from 98 under selection (b), and 247 from 76 and 982 from 99 under (a), against the paper's 47
from 35 and 658 from 94. Under O9 (a), selection (b) gives no p < 0.05 (L2/3 -0.013, p = 0.41; L5
-0.0017, p = 0.66; 210 cells from 87 and 1108 from 138), while selection (a) gives L2/3 omission FSB
higher when running with O4 a (0.034, p = 0.0048), which would not reproduce the claim.

### fig7d-running-lowers-omission-rate

> During omission, firing rate is lower when running than when stationary, in L2/3 and L5.

Expected L2/3 -0.275 (0.175), L5 -0.11 (0.05). Cells as in fig7c, under its readings.

In [ ]:
rows = []
for cl, D in LOC.items():
    for w in ["250", "750"]:
        for sel in "ab":
            for lay in ["L2/3", "L5"]:
                d = om_cells(D[D.layer == lay], w, sel)
                d["y"] = vbn.mod_index(d[f"om_run_{w}_fr"], d[f"om_stat_{w}_fr"])
                rows.append(fit(d, cells=cl, O7=w, selection=sel, layer=lay))
T7D = show(rows)
tol = {"L2/3": (-0.275, 0.175), "L5": (-0.11, 0.05)}
T7D["within"] = [abs(c - tol[l][0]) <= tol[l][1] for c, l in zip(T7D.coef, T7D.layer)]
T7D["neg_sig"] = (T7D.coef < 0) & (T7D.p < 0.05)
print(T7D.groupby(["cells", "O7", "selection"]).agg(both_neg_sig=("neg_sig", "all"), both_within=("within", "all")).to_string())
for cl in ["all", "opto"]:
    d = om_cells(LOC[cl][LOC[cl].layer == "L5"], "250", "a")
    print(cl, "L5, selection a: cells with no omission spikes in one state (index -1 or +1):",
          int(((d.om_run_250_spk == 0) ^ (d.om_stat_250_spk == 0)).sum()), "of", len(d))

**Result: not reproduced.** Under selection (b), L2/3 omission firing is lower when running (O9 a
-0.146, p = 0.0002; O9 b -0.156, p = 0.0006; within tolerance) but L5 is not (O9 a -0.019, p = 0.29;
O9 b -0.003, p = 0.88); 750 ms gives the same. Under selection (a) both indices are positive (O9 a:
L2/3 0.252, L5 0.148; O9 b: 0.229 and 0.144), driven by cells with no omission spikes in one state
(272 of 1407 and 187 of 982 L5 cells).

## Summary

| Claim | Outcome |
|---|---|
| model-positive-error-short-latency | partially reproduced (seed-sensitive) |
| model-negative-error-delayed-release | reproduced |
| fig2a-weak-sensory-silenced | not reproduced |
| fig2b-breakthrough-doublet-threshold | reproduced |
| fig2c-robust-to-input-delay | reproduced |
| fig2d-latency-grows-as-sensation-weakens | reproduced |
| fig3a-low-gain-suppresses-errors | reproduced |
| fig3b-gain-lowers-burst-threshold | reproduced |
| fig3c-gain-keeps-release-delay | not reproduced |
| fig4a-basket-reduction-lowers-burst-threshold | reproduced |
| fig4b-chc-reduction-shortens-latency | reproduced (assumption) |
| fig4cd-l5it-shortens-release-and-doublets | reproduced |
| fig4d-strong-prediction-lowers-burst-threshold | reproduced |
| fig4ef-compensation-restores-latency | reproduced |
| ccf-layer-link (link) | reproduced |
| cell-counts | reproduced |
| fig5c-fsb-highest-on-first-repetition | reproduced (at the edge of the tolerance) |
| fig5c-l5-fsb-keeps-declining | reproduced |
| fig5c-l23-fsb-stable | reproduced |
| fig5d-firing-rate-dips-then-stable | reproduced |
| figs1-post-omission-bursting | reproduced |
| figs1-novel-image-bursting | reproduced |
| figs1-novel-firing-rate-unchanged | not reproduced |
| fig6b-omission-firing-below-spontaneous | partially reproduced (assumption) |
| fig6c-omission-bursting-below-spontaneous | partially reproduced (assumption) |
| figs2-l23-l5-activity-correlated | reproduced |
| figs2-l5-bursting-not-predicted-by-rate | not reproduced |
| fig7a-running-raises-first-repetition-bursting | reproduced |
| fig7b-running-raises-first-repetition-rate | partially reproduced |
| fig7c-running-leaves-omission-bursting | reproduced (assumption) |
| fig7d-running-lowers-omission-rate | not reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

How the analysis differs from the paper's. The model is integrated in NumPy from the Methods'
equations, not in Brian 2, and the paper's code was not available; L4, ChC and L5 IT ranges for the
model figures are plotting choices. On the recordings, which sessions enter is open (O9 of the
claims file); all 153 sessions give about 35% more cells than the paper, and the paper's counts are
matched only by the 104 sessions of the Sst-Cre and Vip-Cre Ai32 mice. Every claim is run under both,
and on all 16 readings of O1 to O3; no outcome now depends on O9, and fig5c-fsb-highest-on-first-repetition
is within tolerance only on 2 of the 16 selections under O9 (b), by 0.0003 at most, under a reading
that codes repetition as k (8 selections under O9 a, by up to 0.0012).
The spontaneous period of Figure 6 is taken as both spontaneous blocks, an assumption; the 10 s grey
screen right after the task alone, the literal reading of "immediately following", gives a different
outcome for fig6b and fig6c. Under D7, 26 of the 104 O9 (b) sessions have no stationary Repetition 1
presentation, so Figure 7A and 7B cannot include every cell, as the paper's degrees of freedom imply.
Mixed models that did not converge were refitted with other optimizers; with the converged fit the
novel against familiar L5 rate difference is significant under both O9 readings. Figures S1 and S2 were not
in the text, so their tests are read from the Methods.

What was found about the dataset. The version holds 153 sessions from 81 mice (104 from Sst-Cre or
Vip-Cre mice with ChR2, 49 wild type), with spikes, stimulus tables, running speed and optotagging
pulses as the paper describes. Units carry no layer; the CCFv3 coordinates of their peak channels
place 7996 of 7997 VISp units in VISp, but 351 VISp units in 7 sessions have all coordinates 0. The
spontaneous block that follows the task lasts 10 s in every session and starts 18 to 29 s after it,
except in one wild-type session, where it starts 3277 s after the last task presentation; the
289 s spontaneous block starts about 940 s after the task. Unit and electrode columns have no descriptions in the
files. The clock check separates images from omissions in most sessions (median AUC 0.95 for the
count 30 to 100 ms after onset, 0.50 before onset), but in 10 sessions the AUC is below 0.8 (lowest
0.55), and 2 sessions have no VISp unit to check; for these the shared clock is not established here.